
# Reliable Explanation-Aware Intrusion Detection under Drift and Adversarial Evasion
## Six-Pillar Pipeline — Validation-Optimized Selective Attribution Edition

This notebook evaluates one validation-derived and subsequently frozen explanation-guided policy across six experimental pillars:

1. **Host-based natural concept drift**
2. **Temporal natural concept drift**
3. **Hard concept drift**
4. **Zero-day attack holdout**
5. **Constrained black-box adversarial evasion**
6. **Multi-seed robustness**

### Final deployment architecture

The final system uses **selective explanation verification**:

**Calibrated LightGBM probability inference → validation-optimized confidence screen → selective SHAP/cosine verification → three-tier routing**

- **Direct confidence resolution:** only validation-approved high-confidence benign or malicious flows bypass SHAP.
- **Selective explanation verification:** all unresolved flows receive predicted-class SHAP attribution and cosine similarity verification.
- **Tier 1 — ALLOW:** trusted benign traffic.
- **Tier 2 — BLOCK:** trusted malicious traffic.
- **Tier 3 — ESCALATE:** unresolved traffic requiring administrative review.

### Validation-only optimization

Two optimization stages are learned from the validation partition only:

1. The explanation-guided routing thresholds are selected subject to the predefined ANMR and FBR safety constraints.
2. The selective-attribution confidence screen is then optimized to minimize the fraction of flows requiring SHAP while preserving those same validation safety constraints.

All thresholds are frozen before Host-Test, Time-Test, Hard-Test, zero-day, adversarial, and multi-seed evaluation.

### Throughput methodology

The original **100%-SHAP policy** is retained only as an experimental safety/throughput baseline. The proposed final architecture is the selective-attribution policy, allowing the manuscript to quantify whether reduced explanation invocation improves throughput without weakening ANMR or FBR.


In [ ]:
# BLOCK 1: GLOBAL VARIABLE INITIALIZATION & CACHE PURGE ENGINE
import gc
import sys

print("=" * 115)
print("[Cache Purge Engine] Initializing clean-room memory state for the pipeline...")
print("=" * 115)
sys.stdout.flush()

# 1. Compile an inventory of every critical pipeline global variable name
pipeline_variables = [
    # Data Frame Components
    "df_raw",
    "df_clean",
    "df_drift_pool",
    "df_zero_day_isolated",
    "train_df",
    "val_df",
    "host_test_df",
    "time_test_df",
    "hard_test_df",
    "true_external_df",
    # NumPy Matrix Tensors
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_diag",
    "y_diag",
    "X_eval",
    "y_eval",
    "X_zday",
    "y_zday",
    "X_transfer_numpy",
    "y_transfer_numpy",
    # Model & Calibration Collections
    "lgb_model",
    "loop_model",
    "calibrators",
    "loop_calibrators",
    "explainer",
    "explainer_diag",
    "loop_explainer",
    "train_signatures",
    "train_variances",
    "loop_signatures",
    # Optimization Parameters & Global Telemetry
    "TAU_P",
    "FINAL_TAU_P",
    "TAU_S",
    "FINAL_TAU_S",
    "comprehensive_metrics_log",
    "block_12_records",
    "block_13_records",
    "block_14_records",
    "SELECTIVE_BENIGN_DIRECT_PROB_TAU",
    "SELECTIVE_BENIGN_DIRECT_MARGIN_TAU",
    "SELECTIVE_ATTACK_DIRECT_PROB_TAU",
    "SELECTIVE_ATTACK_DIRECT_MARGIN_TAU",
    "SELECTIVE_VALIDATION_RESULTS_DF",
    "SELECTIVE_VALIDATION_BEST_ROW",
    "SELECTIVE_ATTRIBUTION_ABLATION_DF",
    "multi_seed_results_df",
]

# 2. Iteratively evict old pointers from the global namespace if they exist
purged_count = 0
for var in pipeline_variables:
    if var in globals():
        del globals()[var]
        purged_count += 1
    if var in locals():
        del locals()[var]

# 3. Explicitly reset your foundational global tracking matrices to crisp baseline defaults
global partition_caches, comprehensive_metrics_log

partition_caches = {}  # Wipes out all Stage 2 ablation row caches
    # Evaporates obsolete evaluation caches
comprehensive_metrics_log = (
    []
)  # Drops old metric arrays to prevent bar plot duplication

# 4. Force immediate aggressive system garbage collection
gc.collect()

print(f"[Purge Complete] Evicted {purged_count} active variables from RAM.")
print("[System Status] Global partition caches re-initialized to: empty dictionary {}")
print("[System Status] Master telemetry log ledger re-initialized to: empty list []")
print("=" * 115 + "\n")
sys.stdout.flush()


In [ ]:
# BLOCK 1B: MOUNT GOOGLE DRIVE & ATTACH DEPENDENCY RUNTIMES
from google.colab import drive
drive.mount('/content/drive')

!pip install shap lightgbm scikit-learn psutil pandas numpy --quiet

import os
import re
import gc
import time
import json
import glob
import psutil
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score, brier_score_loss
from sklearn.calibration import IsotonicRegression
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
RESULTS_DIR = "./behavioral_reliability_perfected_results_main"
os.makedirs(os.path.join(RESULTS_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)

# Central Configuration Object mapped to your exact Google Drive path names
CONFIG = {
    'base_path': '/content/drive/MyDrive',
    'bccc_dir': '/content/drive/MyDrive/BCCC-CIC-IDS-2017',
    'results_dir': '/content/drive/MyDrive/IDS_XAI_Project/',
    'keep_benign_files': {'friday_benign.csv', 'thursday_benign.csv'},
    'min_class_count': 30,
    'rare_classes_expected': {'Heartbleed', 'Web_SQL_Injection'},
    'zero_day_class': 'Web_Brute_Force',
    'max_rows_per_class': 500
}
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print("[Initialization] Virtual execution environments established and dependencies fixed.")

# Low-latency operational controls
CONFIG.setdefault("attribution_batch_size", 8192)
CONFIG.setdefault("prediction_batch_size", 65536)
CONFIG.setdefault("adversarial_epsilons", [0.00, 0.01, 0.025, 0.05, 0.10])
CONFIG.setdefault("adversarial_random_candidates", 4)
CONFIG.setdefault("adversarial_eval_max_rows", 20000)


In [ ]:
# BLOCK 2: GLOBAL LABEL SCHEMA, THREE-TIER ROUTING POLICY,
#          AND LOW-LATENCY EXPLANATION HELPERS

import numpy as np
import shap


# =============================================================================
# 1. GLOBAL AND CANONICAL LABEL SCHEMAS
# =============================================================================

GLOBAL_CLASS_INDEX = {
    "Benign": 0,
    "Botnet_ARES": 1,
    "DDoS_LOIT": 2,
    "DoS_GoldenEye": 3,
    "DoS_Hulk": 4,
    "DoS_Slowhttptest": 5,
    "DoS_Slowloris": 6,
    "FTP-Patator": 7,
    "Port_Scan": 8,
    "SSH-Patator": 9,
    "Web_Brute_Force": 10,
    "Web_XSS": 11,
}
REVERSE_CLASS_INDEX = {index: name for name, index in GLOBAL_CLASS_INDEX.items()}

CANONICAL_LABEL_MAP = {
    "BENIGN": "Benign", "Benign": "Benign", "benign": "Benign",
    "Bot": "Botnet_ARES", "Botnet_ARES": "Botnet_ARES", "BotnetARES": "Botnet_ARES",
    "DDoS": "DDoS_LOIT", "DDOS": "DDoS_LOIT", "DDoS_LOIT": "DDoS_LOIT",
    "DoS_Hulk": "DoS_Hulk", "Hulk": "DoS_Hulk",
    "DoS_attacks_Hulk": "DoS_Hulk", "DoS attacks-Hulk": "DoS_Hulk",
    "DoS_GoldenEye": "DoS_GoldenEye", "DoS_Golden_Eye": "DoS_GoldenEye",
    "DoS attacks-GoldenEye": "DoS_GoldenEye", "DoS_attacks_GoldenEye": "DoS_GoldenEye",
    "DoS_slowloris": "DoS_Slowloris", "DoS_Slowloris": "DoS_Slowloris",
    "DoS attacks-Slowloris": "DoS_Slowloris", "DoS_attacks_Slowloris": "DoS_Slowloris",
    "DoS_Slowhttptest": "DoS_Slowhttptest", "DoS_slowhttptest": "DoS_Slowhttptest",
    "DoS attacks-SlowHTTPTest": "DoS_Slowhttptest",
    "DoS_attacks_SlowHTTPTest": "DoS_Slowhttptest",
    "FTP_Patator": "FTP-Patator", "FTP-Patator": "FTP-Patator",
    "FTPPatator": "FTP-Patator", "FTP-BruteForce": "FTP-Patator",
    "FTP_BruteForce": "FTP-Patator",
    "SSH_Patator": "SSH-Patator", "SSH-Patator": "SSH-Patator",
    "SSHPatator": "SSH-Patator", "SSH-Bruteforce": "SSH-Patator",
    "SSH_Bruteforce": "SSH-Patator",
    "PortScan": "Port_Scan", "Port_Scan": "Port_Scan", "Portscan": "Port_Scan",
    "Web_Attack_Brute_Force": "Web_Brute_Force",
    "Web_Attack__Brute_Force": "Web_Brute_Force",
    "Web_Brute_Force": "Web_Brute_Force",
    "Brute Force -Web": "Web_Brute_Force", "Brute_Force_Web": "Web_Brute_Force",
    "Web_Attack_XSS": "Web_XSS", "Web_Attack__XSS": "Web_XSS",
    "Web_XSS": "Web_XSS", "Brute Force -XSS": "Web_XSS",
    "Brute_Force_XSS": "Web_XSS",
    # The existing experimental taxonomy merges SQL injection into Web_XSS.
    "Web_Attack_Sql_Injection": "Web_XSS",
    "Web_Attack__Sql_Injection": "Web_XSS",
    "Web_SQL_Injection": "Web_XSS", "SQL Injection": "Web_XSS",
    "SQL_Injection": "Web_XSS",
}


# =============================================================================
# 2. ACTIVE TRAINING SPACE AND ZERO-DAY SENTINEL
# =============================================================================

ZERO_DAY_CLASS_NAME = CONFIG.get("zero_day_class", "Web_Brute_Force")
ACTIVE_CLASS_NAMES = [
    name for name in GLOBAL_CLASS_INDEX if name != ZERO_DAY_CLASS_NAME
]
ACTIVE_CLASS_INDEX = {
    name: index for index, name in enumerate(ACTIVE_CLASS_NAMES)
}
ACTIVE_REVERSE_CLASS_INDEX = {
    index: name for name, index in ACTIVE_CLASS_INDEX.items()
}
CLASS_COUNT = len(ACTIVE_CLASS_NAMES)
BENIGN_IDX = ACTIVE_CLASS_INDEX["Benign"]
ZERO_DAY_ENCODED_LABEL = -1


def encode_active_labels(label_series, zero_day_value=ZERO_DAY_ENCODED_LABEL):
    """Encode known labels into 0..CLASS_COUNT-1 and the held-out class as -1."""
    mapped = label_series.map(ACTIVE_CLASS_INDEX)
    if zero_day_value is not None:
        mapped = mapped.fillna(zero_day_value)
    if mapped.isna().any():
        unknown = sorted(label_series[mapped.isna()].astype(str).unique().tolist())
        raise ValueError(f"Unrecognized labels in active label space: {unknown}")
    return mapped.astype(np.int32).to_numpy()


def resolve_active_class_name(index_value):
    """Resolve an active class index or the zero-day sentinel to a class name."""
    try:
        index_value = int(index_value)
    except (TypeError, ValueError):
        return f"Unknown_Class_{index_value}"
    if index_value == ZERO_DAY_ENCODED_LABEL:
        return ZERO_DAY_CLASS_NAME
    return ACTIVE_REVERSE_CLASS_INDEX.get(
        index_value, f"Unknown_Class_{index_value}"
    )


# Validation-selected values are assigned in Block 10.
STRICT_BENIGN_PROB_TAU = None
STRICT_BENIGN_SIM_TAU = None


def _resolve_threshold(explicit_value, global_name, fallback_value):
    if explicit_value is not None:
        return float(explicit_value)
    global_value = globals().get(global_name)
    return float(fallback_value if global_value is None else global_value)


# =============================================================================
# 3. FINAL THREE-TIER POLICY
# =============================================================================

def route_policy_batch(
    pred_names,
    true_names,
    attack_confidence,
    similarities,
    tau_p,
    tau_s,
    benign_prob_tau=None,
    benign_sim_tau=None,
    override_prob_tau=None,
    override_sim_floor=None,
):
    """Apply the frozen harmonized three-tier policy to a batch of flows."""

    pred_names = np.asarray(pred_names, dtype=object)
    true_names = np.asarray(true_names, dtype=object)
    attack_confidence = np.asarray(
        attack_confidence,
        dtype=float,
    )
    similarities = np.asarray(
        similarities,
        dtype=float,
    )

    n_samples = len(pred_names)

    if not (
        len(true_names)
        == len(attack_confidence)
        == len(similarities)
        == n_samples
    ):
        raise ValueError(
            "All routing arrays must have equal length."
        )

    if not np.all(np.isfinite(attack_confidence)):
        raise ValueError(
            "attack_confidence contains non-finite values."
        )

    if not np.all(np.isfinite(similarities)):
        raise ValueError(
            "similarities contains non-finite values."
        )

    # --------------------------------------------------------------
    # Resolve all frozen thresholds.
    # --------------------------------------------------------------
    tau_p = float(tau_p)
    tau_s = float(tau_s)

    tau_b = _resolve_threshold(
        benign_prob_tau,
        "STRICT_BENIGN_PROB_TAU",
        tau_p,
    )

    tau_bs = _resolve_threshold(
        benign_sim_tau,
        "STRICT_BENIGN_SIM_TAU",
        tau_s,
    )

    tau_override_prob = _resolve_threshold(
        override_prob_tau,
        "STRICT_OVERRIDE_PROB_TAU",
        0.9995,
    )

    tau_override_sim = _resolve_threshold(
        override_sim_floor,
        "STRICT_OVERRIDE_SIM_FLOOR",
        0.02,
    )

    # Ordinary benign release must satisfy both attack similarity
    # and strict benign similarity requirements.
    effective_benign_similarity = max(
        tau_s,
        tau_bs,
    )

    # Predicted-malicious flows use the original malicious
    # explanation-similarity requirement.
    malicious_similarity_threshold = (
        0.5 * tau_s
    )

    predicted_benign = pred_names == "Benign"
    true_benign = true_names == "Benign"

    benign_probability = (
        1.0 - attack_confidence
    )

    # --------------------------------------------------------------
    # TIER 1A: Ordinary benign allow.
    # --------------------------------------------------------------
    ordinary_benign_allow = (
        predicted_benign
        & (
            benign_probability
            >= tau_b
        )
        & (
            similarities
            >= effective_benign_similarity
        )
    )

    # --------------------------------------------------------------
    # TIER 1B: High-confidence benign override.
    #
    # This permits a predicted-benign flow to bypass the ordinary
    # similarity threshold only when:
    #
    # 1. benign probability is exceptionally high; and
    # 2. similarity remains above the validation-selected safety floor.
    # --------------------------------------------------------------
    high_confidence_benign_override = (
        predicted_benign
        & (
            benign_probability
            >= tau_override_prob
        )
        & (
            similarities
            >= tau_override_sim
        )
    )

    tier_1_allow = (
        ordinary_benign_allow
        | high_confidence_benign_override
    )

    # --------------------------------------------------------------
    # TIER 2: Autonomous block.
    # --------------------------------------------------------------
    tier_2_block = (
        (~predicted_benign)
        & (
            attack_confidence
            >= tau_p
        )
        & (
            similarities
            >= malicious_similarity_threshold
        )
    )

    # --------------------------------------------------------------
    # TIER 3: Everything unresolved is escalated.
    # --------------------------------------------------------------
    tier_3_escalate = ~(
        tier_1_allow
        | tier_2_block
    )

    actions = np.full(
        n_samples,
        "ESCALATE",
        dtype=object,
    )

    tiers = np.full(
        n_samples,
        "Tier 3",
        dtype=object,
    )

    actions[tier_1_allow] = "ALLOW"
    tiers[tier_1_allow] = "Tier 1"

    actions[tier_2_block] = "BLOCK"
    tiers[tier_2_block] = "Tier 2"

    return {
        "actions": actions,
        "tiers": tiers,

        "tier_1_allow": tier_1_allow,
        "tier_2_block": tier_2_block,
        "tier_3_escalate": tier_3_escalate,
        "human_escalated": tier_3_escalate,

        "ordinary_benign_allow":
            ordinary_benign_allow,

        "high_confidence_benign_override":
            high_confidence_benign_override,

        "missed_attack":
            (~true_benign) & tier_1_allow,

        "false_block":
            true_benign & tier_2_block,

        "false_alarm":
            true_benign & tier_3_escalate,

        "benign_probability":
            benign_probability,

        "effective_benign_similarity_threshold":
            effective_benign_similarity,

        "malicious_similarity_threshold":
            malicious_similarity_threshold,

        "override_probability_threshold":
            tau_override_prob,

        "override_similarity_floor":
            tau_override_sim,
    }


def shared_policy_route(
    pred_name,
    true_name,
    attack_confidence,
    similarity,
    tau_p,
    tau_s,
    benign_prob_tau=None,
    benign_sim_tau=None,
):
    """Single-flow wrapper retained for diagnostics and unit tests."""
    routed = route_policy_batch(
        np.asarray([pred_name], dtype=object),
        np.asarray([true_name], dtype=object),
        np.asarray([attack_confidence], dtype=float),
        np.asarray([similarity], dtype=float),
        tau_p,
        tau_s,
        benign_prob_tau,
        benign_sim_tau,
    )
    return {
        "action": routed["actions"][0],
        "tier": routed["tiers"][0],
        "missed_attack": bool(routed["missed_attack"][0]),
        "false_block": bool(routed["false_block"][0]),
        "false_alarm": bool(routed["false_alarm"][0]),
        "escalated": bool(routed["human_escalated"][0]),
        "benign_probability": float(routed["benign_probability"][0]),
    }


# =============================================================================
# 4. LOW-LATENCY ATTRIBUTION AND SIGNATURE SIMILARITY
# =============================================================================

def fast_predicted_class_attributions(
    model,
    X_matrix,
    predicted_indices,
    batch_size=8192,
):
    """Extract predicted-class LightGBM contributions in vectorized batches."""
    X_matrix = np.asarray(X_matrix, dtype=np.float32)
    predicted_indices = np.asarray(predicted_indices, dtype=np.int32)

    if X_matrix.ndim != 2 or predicted_indices.ndim != 1:
        raise ValueError("X_matrix must be 2-D and predicted_indices must be 1-D.")
    if len(X_matrix) != len(predicted_indices):
        raise ValueError("X_matrix and predicted_indices must align.")
    if np.any((predicted_indices < 0) | (predicted_indices >= CLASS_COUNT)):
        raise ValueError("predicted_indices contains an out-of-range class index.")

    n_rows, n_features = X_matrix.shape
    output = np.zeros((n_rows, n_features), dtype=np.float32)
    batch_size = max(1, int(batch_size))

    for start in range(0, n_rows, batch_size):
        end = min(start + batch_size, n_rows)
        X_batch = X_matrix[start:end]
        p_batch = predicted_indices[start:end]

        try:
            contrib = np.asarray(
                model.predict(X_batch, pred_contrib=True)
            )
            if (
                contrib.ndim == 2
                and contrib.shape[1] == CLASS_COUNT * (n_features + 1)
            ):
                contrib = contrib.reshape(
                    len(X_batch), CLASS_COUNT, n_features + 1
                )
                batch_output = contrib[
                    np.arange(len(X_batch)), p_batch, :-1
                ]
            elif contrib.ndim == 3 and contrib.shape[1] == CLASS_COUNT:
                batch_output = contrib[
                    np.arange(len(X_batch)), p_batch, :-1
                ]
            elif contrib.ndim == 3 and contrib.shape[2] == CLASS_COUNT:
                batch_output = contrib[
                    np.arange(len(X_batch)), :-1, p_batch
                ]
            else:
                raise ValueError(
                    f"Unexpected native contribution shape: {contrib.shape}"
                )
        except Exception as native_error:
            try:
                raw = shap.TreeExplainer(model).shap_values(X_batch)
                if isinstance(raw, list):
                    batch_output = np.vstack(
                        [raw[p_batch[row]][row] for row in range(len(X_batch))]
                    )
                else:
                    raw = np.asarray(raw)
                    if raw.ndim == 3 and raw.shape[1] == n_features:
                        batch_output = raw[
                            np.arange(len(X_batch)), :, p_batch
                        ]
                    elif raw.ndim == 3 and raw.shape[2] == n_features:
                        batch_output = raw[
                            np.arange(len(X_batch)), p_batch, :
                        ]
                    elif raw.shape == (len(X_batch), n_features):
                        batch_output = raw
                    else:
                        raise ValueError(
                            f"Unexpected SHAP fallback shape: {raw.shape}"
                        )
            except Exception as shap_error:
                raise RuntimeError(
                    "Native LightGBM contributions and SHAP fallback both failed."
                ) from shap_error

        output[start:end] = np.asarray(batch_output, dtype=np.float32)

    return output


def compute_predicted_signature_similarity(
    model,
    reference_signatures,
    X_matrix,
    predicted_indices,
    predicted_names,
    batch_size=8192,
):
    """Compute cosine similarity to each flow's predicted-class signature."""
    X_matrix = np.asarray(X_matrix, dtype=np.float32)
    predicted_indices = np.asarray(predicted_indices, dtype=np.int32)
    predicted_names = np.asarray(predicted_names, dtype=object)

    if not (
        len(X_matrix) == len(predicted_indices) == len(predicted_names)
    ):
        raise ValueError("Attribution inputs must contain the same number of rows.")

    attributions = fast_predicted_class_attributions(
        model, X_matrix, predicted_indices, batch_size=batch_size
    )
    norms = np.linalg.norm(attributions, axis=1, keepdims=True)
    normalized = np.divide(
        attributions,
        norms,
        out=np.zeros_like(attributions, dtype=np.float32),
        where=norms > 1e-12,
    )

    similarities = np.zeros(len(X_matrix), dtype=np.float32)
    for class_name in np.unique(predicted_names):
        mask = predicted_names == class_name
        signature = reference_signatures.get(class_name)
        if signature is None:
            continue
        signature = np.asarray(signature, dtype=np.float32)
        if signature.shape != (X_matrix.shape[1],):
            raise ValueError(
                f"Signature for {class_name} has shape {signature.shape}; "
                f"expected {(X_matrix.shape[1],)}."
            )
        signature_norm = np.linalg.norm(signature)
        if signature_norm > 1e-12:
            similarities[mask] = normalized[mask] @ (
                signature / signature_norm
            )

    return similarities, normalized


print(
    f"[Active Label Space] num_class={CLASS_COUNT}; "
    f"zero-day={ZERO_DAY_CLASS_NAME}; active classes={ACTIVE_CLASS_NAMES}"
)
print(
    "[Initialization] Three-tier routing and batched attribution helpers locked."
)


In [ ]:
# BLOCK 3: STRATIFIED PIPELINE INGESTION CORE (RARE CLASS FILTERS REMOVAL ENFORCED)
def clean_colname(c):
    c = str(c).strip().replace('\ufeff', '')
    c = re.sub(r'[^0-9a-zA-Z]+', '_', c)
    c = re.sub(r'_+', '_', c).strip('_').lower()
    return c

def normalize_columns(df):
    df.columns = [clean_colname(c) for c in df.columns]
    return df

def standardize_label_text(x):
    if pd.isna(x): return "Benign"
    s = str(x).strip().replace('–', '-').replace('—', '-').replace(' ', '_').replace('-', '_')
    s = re.sub(r'_+', '_', s)
    return s

def normalize_labels(df):
    candidates = ['label', 'class', 'attack', 'attack_cat', 'category']
    lbl_col = None
    for c in candidates:
        if c in df.columns:
            lbl_col = c
            break
    if lbl_col is None:
        df['label'] = 'Benign'
        return df
    df.rename(columns={lbl_col: 'label'}, inplace=True)
    df['label'] = df['label'].astype(str).str.strip().map(
        lambda x: CANONICAL_LABEL_MAP.get(x, CANONICAL_LABEL_MAP.get(standardize_label_text(x), 'Benign'))
    )
    return df

def load_bccc_balanced_workspace():
    print(f"[Ingestion Loop] Aggregating matrices inside path: {CONFIG['bccc_dir']}")
    csv_files = sorted(glob.glob(os.path.join(CONFIG['bccc_dir'], "**", "*.csv"), recursive=True))

    if len(csv_files) == 0:
        print("[Ingestion Fallback] Target folder empty. Deploying standalone matrix simulation...")
        np.random.seed(RANDOM_STATE)
        n_samples = 35000
        features = list(FEATURE_MAP_CIC.values()) + ['active_mean', 'active_max', 'idle_mean', 'idle_max']
        data = {f: np.random.exponential(scale=15.0, size=n_samples) for f in features}
        data['src_ip'] = np.random.choice([f"192.168.10.{i}" for i in range(1, 150)], size=n_samples)
        data['label'] = np.random.choice(list(GLOBAL_CLASS_INDEX.keys()), size=n_samples, p=[0.3] + [0.7/11]*11)
        df = pd.DataFrame(data)
        # Enforce dynamic rare class extraction drop on synthetic generation too
        df = df[~df['label'].isin(CONFIG['rare_classes_expected'])]
        return df, features

    parts = []
    for f in csv_files:
        fname = os.path.basename(f).lower()
        # Native class imbalance control layer execution
        if 'benign' in fname and fname not in CONFIG['keep_benign_files']:
            continue
        try:
            df = pd.read_csv(f, nrows=12000, low_memory=False)
            df = normalize_columns(df)
            df = normalize_labels(df)

            # CRITICAL CORRECTION: Explicitly exclude rare classes below min threshold count limits
            df = df[~df['label'].isin(CONFIG['rare_classes_expected'])]
            parts.append(df)
        except Exception as e:
            continue

    master_df = pd.concat(parts, ignore_index=True, sort=False)

    # Secondary programmatic safe verification trap
    for rare_cls in CONFIG['rare_classes_expected']:
        master_df = master_df[master_df['label'] != rare_cls]

    features = [c for c in master_df.columns if c != 'label' and c not in ['src_ip', 'dst_ip', 'timestamp']]
    return master_df, features

df_raw, original_features = load_bccc_balanced_workspace()
print("[Ingestion Engine] Historical analytical core shape (Outliers Dropped):", df_raw.shape)
print("[Ingestion Engine] Current Class Populations Summary:\n", df_raw['label'].value_counts())


In [ ]:
# BLOCK 4: STAGE 1 — MEMORIZATION AND TOPOLOGY LEAKAGE ELIMINATION
def behavior_focused_clean(df, continuous_features):
    print("[Stage 1] Evaporating routing metadata fields to enforce behavioral splits...")

    # Clean out extreme value floats from target structures
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=['label'])

    # FIXED: True exact administrative, contextual, and tracking leakage columns
    leakage_cols = [
        "flow_id", "timestamp", "Date", "src_ip", "dst_ip",
        "HostID", "label", "source_file", "src_port", "dst_port"
    ]

    # Convert leakage elements to lowercase for robust case-insensitive alignment protection
    leakage_lower = [str(col).lower() for col in leakage_cols]

    # Isolate valid, invariant behavioral features while systematically scrubbing leakage channels
    valid_features = [
        f for f in continuous_features
        if f in df.columns and str(f).lower() not in leakage_lower
    ]

    # Enforce numeric integrity and impute missing frames using baseline medians
    for col in valid_features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Handle cases where columns might be empty or all-NaN gracefully
        if df[col].notna().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(0.0)

    return df, valid_features

# Execute clean, data-leakage-free sanitization across the active dataframe workspace
df_clean, expected_features = behavior_focused_clean(df_raw, original_features)
print(f"[Stage 1 Complete] Invariant feature attributes isolated count: {len(expected_features)}")


In [ ]:
# BLOCK 5: STAGE 1.5 — ZERO-DAY HOLDOUT QUARANTINE CORE (PRE-SPLIT ISOLATION)
import sys

print("=" * 115)
print(f"[Quarantine Layer] Isolating {CONFIG['zero_day_class']} from master dataframe...")
print("=" * 115)
sys.stdout.flush()

# 1. Enforce physical separation before any data partitioning occurs
df_clean = df_clean.copy().reset_index(drop=True)
df_zero_day_isolated = df_clean[df_clean['label'] == CONFIG['zero_day_class']].copy().reset_index(drop=True)

# 2. Extract and preserve the clean pool containing ONLY the remaining 11 classes
df_drift_pool = df_clean[df_clean['label'] != CONFIG['zero_day_class']].copy().reset_index(drop=True)

# 3. Print out verification logs showing exactly which classes remain for splitting
print(f"\n[Verification] {CONFIG['zero_day_class']} successfully removed.")
print(f"[*] Quarantined Zero-Day Sample Points: {len(df_zero_day_isolated):,}")
print(f"[*] Remaining Data Pool Size for Splitting: {len(df_drift_pool):,} rows")

print("\n" + "-"*50)
print("REMAINING ACTIVE CLASSES PASSED TO SPLITTING ENGINE:")
print("-"*50)
remaining_counts = df_drift_pool['label'].value_counts()
print(remaining_counts.to_string())
print("-" * 50 + "\n")
sys.stdout.flush()

# Quick structural assertion to prevent code from executing if isolation fails
assert CONFIG['zero_day_class'] not in df_drift_pool['label'].unique(), "CRITICAL FAULT: Zero-day class leaked!"


In [ ]:
import sys


# BLOCK 6: STAGE 2 — COVARIATE DRIFT STRATIFIED ABLATION STUDY ENGINE
def run_host_ablation_study(df, expected_features):
    # Declare global at the very beginning of the function scope to prevent SyntaxError
    global partition_caches

    print("=" * 115)
    print("[Stage 2] Launching Proportional Stratified Host-Ratio Ablation Study Engine...")
    print("=" * 115)
    sys.stdout.flush()

    df = df.copy().reset_index(drop=True)
    if len(df) == 0:
        print("CRITICAL ERROR: Input DataFrame is empty!")
        sys.stdout.flush()
        return 0.5

    df['hostid_str'] = df['src_ip'].astype(str).map(lambda x: x.split('.')[-1] if '.' in x else '1')
    global_total_labels = df['label'].nunique()

    ratios = [0.7, 0.6, 0.5, 0.4]
    ablation_records = []
    partition_caches = {}

    for r in ratios:
        train_list, val_list, host_list, time_list, hard_list = [], [], [], [], []

        # Determine strict distribution bounds dynamically based on the ablation ratio
        unseen_target = round(1.0 - r, 1)

        for label_name, group in df.groupby('label'):
            group_len = len(group)
            shuffled_group = group.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

            # Map proportional cutoffs matching the specific ratio window safely
            f_train = max(1, int(group_len * (r * 0.8)))
            f_val   = max(1, int(group_len * r))
            f_host  = max(1, int(group_len * (r + unseen_target * 0.35)))
            f_time  = max(1, int(group_len * (r + unseen_target * 0.70)))

            train_list.append(shuffled_group.iloc[:f_train])
            val_list.append(shuffled_group.iloc[f_train:f_val])
            host_list.append(shuffled_group.iloc[f_val:f_host])
            time_list.append(shuffled_group.iloc[f_host:f_time])
            hard_list.append(shuffled_group.iloc[f_time:])

        df_train_p = pd.concat(train_list, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        df_val_p   = pd.concat(val_list, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        df_host_p  = pd.concat(host_list, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        df_time_p  = pd.concat(time_list, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        df_hard_p  = pd.concat(hard_list, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

        # Label Loss evaluates to 0 globally because stratification shields minority blocks
        val_loss = max(0, global_total_labels - df_val_p['label'].nunique())
        time_loss = max(0, global_total_labels - df_time_p['label'].nunique())

        rec = {
            'Seen Ratio': r, 'Unseen Ratio': unseen_target,
            'Val Rows': len(df_val_p),   'Val Hosts': df_val_p['hostid_str'].nunique(),   'Val Labels': df_val_p['label'].nunique(),
            'Host Rows': len(df_host_p),  'Host Hosts': df_host_p['hostid_str'].nunique(),  'Host Labels': df_host_p['label'].nunique(),
            'Time Rows': len(df_time_p),  'Time Hosts': df_time_p['hostid_str'].nunique(),  'Time Labels': df_time_p['label'].nunique(),
            'Hard Rows': len(df_hard_p),  'Hard Hosts': df_hard_p['hostid_str'].nunique(),  'Hard Labels': df_hard_p['label'].nunique(),
            'Val Label Loss': val_loss, 'Time Label Loss': time_loss
        }
        ablation_records.append(rec)
        partition_caches[r] = (df_train_p, df_val_p, df_host_p, df_time_p, df_hard_p)

    ablation_df = pd.DataFrame(ablation_records)

    # Print the missing experiment summary matrix tables straight to console
    print("Host split ratio experiment summary:")
    print(ablation_df.drop(columns=['Val Label Loss', 'Time Label Loss']).to_string(index=False))
    print("\n" + "-"*50 + "\n")

    sorted_ablation_df = ablation_df.sort_values(
        by=['Val Label Loss', 'Time Label Loss', 'Val Rows'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    print("Sorted ratio comparison:")
    print(sorted_ablation_df.to_string(index=False))
    print("\n" + "-"*50 + "\n")

    selected_ratio = sorted_ablation_df.loc[0, 'Seen Ratio']
    print(f"Automatically selected host split ratio: Seen/Unseen = {selected_ratio}/{round(1.0 - selected_ratio, 1)}\n")
    sys.stdout.flush()

    t_df, v_df, h_df, tm_df, hd_df = partition_caches[selected_ratio]

    try:
        csv_out_path = os.path.join(CONFIG['results_dir'], 'host_ratio_ablation_study.csv')
        ablation_df.to_csv(csv_out_path, index=False)
    except Exception:
        pass

    print("=" * 115)
    print("                                     FINAL PARALYZED SPLIT SUMMARY                                  ")
    print("=" * 115)

    print(f"\nTrain: rows={len(t_df):,}, hosts={t_df['hostid_str'].nunique()}, labels={t_df['label'].nunique()}")
    print(t_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nValidation: rows={len(v_df):,}, hosts={v_df['hostid_str'].nunique()}, labels={v_df['label'].nunique()}")
    print(v_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nHost-Test: rows={len(h_df):,}, hosts={h_df['hostid_str'].nunique()}, labels={h_df['label'].nunique()}")
    print(h_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nTime-Test: rows={len(tm_df):,}, hosts={tm_df['hostid_str'].nunique()}, labels={tm_df['label'].nunique()}")
    print(tm_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nHard-Test: rows={len(hd_df):,}, hosts={hd_df['hostid_str'].nunique()}, labels={hd_df['label'].nunique()}")
    print(hd_df['label'].value_counts().to_string())
    print("=" * 115)
    sys.stdout.flush()

    return selected_ratio

# Execute code block inside execution cell
# Execute the stratified study directly using the remaining 11 clean classes from Block 5
best_seen_ratio = run_host_ablation_study(df_drift_pool, expected_features)


In [ ]:
# Extract the pristine 11-class stratified splits out of Stage 2's cache
train_df, val_df, host_test_df, time_test_df, hard_test_df = partition_caches[best_seen_ratio]

# Convert stratified splits into position-invariant numpy arrays for LightGBM
X_train = train_df[expected_features].to_numpy()
y_train = encode_active_labels(train_df['label'], zero_day_value=None)

X_val = val_df[expected_features].to_numpy()
y_val = encode_active_labels(val_df['label'], zero_day_value=None)

print(f"[*] Training Target Matrix Dimensions:     {X_train.shape} | Unique Labels: {len(np.unique(y_train))}")
print(f"[*] Validation Target Matrix Dimensions:   {X_val.shape} | Unique Labels: {len(np.unique(y_val))}")
print(f"[*] Data Leakage Check: Is zero-day target inside y_train? -> {ZERO_DAY_ENCODED_LABEL in y_train}")


In [ ]:
import os
# BLOCK 7: STAGE 3 — PRIMARY MODEL SEEDING AND PROBABILITY TUNING
print("[Stage 3] Training invariant tree optimization layer paths...")
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'num_threads': max(1, (os.cpu_count() or 2) - 1),
    'objective': 'multiclass',
    'num_class': CLASS_COUNT,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 32,
    'random_state': RANDOM_STATE,
    'verbose': -1
}

lgb_model = lgb.train(
    params, train_data, num_boost_round=100,
    valid_sets=[val_data], callbacks=[lgb.early_stopping(10, verbose=False)]
)

print("[Calibration Layer] Aligning raw output thresholds with Isotonic Regression...")
val_raw_preds = lgb_model.predict(X_val)
calibrators = {}

for c in range(CLASS_COUNT):
    ir = IsotonicRegression(out_of_bounds='clip')
    target_binary = (y_val == c).astype(int)
    if len(np.unique(target_binary)) < 2:
        target_binary = np.append(target_binary, [0, 1])
        val_raw_preds_padded = np.append(val_raw_preds[:, c], [0.0, 1.0])
        ir.fit(val_raw_preds_padded, target_binary)
    else:
        ir.fit(val_raw_preds[:, c], target_binary)
    calibrators[c] = ir

def get_calibrated_probabilities(model, idx_calibrators, X_mat):
    raw = model.predict(X_mat)
    calibrated = np.zeros_like(raw)
    for cls_idx in range(CLASS_COUNT):
        calibrated[:, cls_idx] = idx_calibrators[cls_idx].transform(raw[:, cls_idx])
    sums = calibrated.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1.0
    return calibrated / sums

print("[Stage 3 Complete] Probability channels normalized successfully.")


In [ ]:
# BLOCK 8: STAGE 4 — SHAP BEHAVIORAL ATTRIBUTION PROFILE ENGINE SIGNATURES (SHAPE CORRECTION FIXED)
def compute_shap_signatures(model, X_ref, df_meta, feature_names):
    print("[Stage 4] Compiling continuous game-theoretic feature consensus metrics...")
    explainer = shap.TreeExplainer(model)

    # Structural density group sampler matching historical sample parameters
    sample_rows = df_meta.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 40), random_state=RANDOM_STATE)
    ).reset_index(drop=True)

    X_sample_numpy = sample_rows[feature_names].to_numpy()
    shap_values = explainer.shap_values(X_sample_numpy)

    signatures = {}
    variance_audits = {}

    for class_name, class_idx in ACTIVE_CLASS_INDEX.items():
        mask = (sample_rows['label'] == class_name).to_numpy()
        if not mask.any(): continue

        # FIXED: Dynamic shape detection handler to insulate against 3D tensor vs List slicing variations
        if isinstance(shap_values, list):
            # Traditional list of arrays format: isolate class list element first, then slice samples
            class_shap = shap_values[class_idx][mask]
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            # Modern 3D NumPy tensor: shap_values shape is (samples, features, classes)
            # Filter samples along axis 0 matching the mask, then isolate the target class index
            class_shap = shap_values[mask, :, class_idx]
        else:
            # Emergency fallback structure encapsulation
            try:
                class_shap = np.array(shap_values)[class_idx][mask]
            except Exception:
                class_shap = shap_values[mask]

        # Row-wise L2 spatial normalization layer
        norms = np.linalg.norm(class_shap, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        normalized_shap = class_shap / norms

        # Extract mean direction consensus blueprint vector
        mean_vector = normalized_shap.mean(axis=0)
        vec_norm = np.linalg.norm(mean_vector)
        signatures[class_name] = mean_vector / (vec_norm if vec_norm > 0 else 1.0)
        variance_audits[class_name] = np.sqrt(normalized_shap.var(axis=0))

    return signatures, variance_audits

# Execute baseline calculations directly inline without wall-of-text print statements
train_signatures, train_variances = compute_shap_signatures(lgb_model, X_train, train_df, expected_features)
print(f"[Stage 4 Complete] Unique Signature vectors generated count: {len(train_signatures)}")


In [ ]:
# BLOCK 9: STAGE 5 — AUTHENTIC SHAP VS LIME CORRELATION INTEGRITY AGREEMENT CHECK

!pip install lime --quiet
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt
import os
import sys

def run_explanation_cross_validation(model, X_matrix, feature_names, df_train_reference):
    print("[Stage 5] Launching True SHAP vs LIME Explanation Cross-Validation Engine...")
    sys.stdout.flush()

    # 1. Initialize True LIME Tabular Explainer over training baseline characteristics
    lime_explainer = LimeTabularExplainer(
        training_data=X_matrix,
        feature_names=feature_names,
        class_names=list(ACTIVE_CLASS_INDEX.keys()),
        mode='classification',
        random_state=RANDOM_STATE
    )

    # 2. Initialize SHAP Explainer
    shap_explainer = shap.TreeExplainer(model)

    sample_size = min(20, len(X_matrix)) # Core sample subset for execution efficiency
    raw_shap_values = shap_explainer.shap_values(X_matrix[:sample_size])

    raw_preds = model.predict(X_matrix[:sample_size])
    predicted_classes = np.argmax(raw_preds, axis=1)

    jaccard_scores = []

    for idx in range(sample_size):
        target_class_idx = predicted_classes[idx]

        # --- A. Isolate True SHAP Top Features ---
        if isinstance(raw_shap_values, np.ndarray) and raw_shap_values.ndim == 3:
            shap_attr = raw_shap_values[idx, :, target_class_idx]
        elif isinstance(raw_shap_values, list):
            shap_attr = raw_shap_values[target_class_idx][idx]
        else:
            shap_attr = raw_shap_values[idx]
        top_shap_features = set(np.argsort(np.abs(shap_attr))[-5:])

        # --- B. Generate True LIME Top Features ---
        predict_fn = lambda x: get_calibrated_probabilities(model, calibrators, x)

        exp = lime_explainer.explain_instance(
            data_row=X_matrix[idx],
            predict_fn=predict_fn,
            num_features=len(feature_names),
            labels=(target_class_idx,)
        )

        local_exp_list = exp.as_map()[target_class_idx]
        top_lime_features = set([feat_idx for feat_idx, weight in sorted(local_exp_list, key=lambda x: abs(x[1]))[-5:]])

        # --- C. Compute Jaccard Intersection Over Union ---
        intersection = len(top_shap_features & top_lime_features)
        union = len(top_shap_features | top_lime_features)
        jaccard_scores.append(intersection / union if union > 0 else 0.0)

    mean_jaccard = np.mean(jaccard_scores)
    print(f"[Stage 5 Metric Verification] Authentic Mean Top-5 Jaccard Agreement: {mean_jaccard:.4f}")
    sys.stdout.flush()

    # =====================================================================
    # VISUALIZATION GENERATION LAYER
    # =====================================================================
    plt.figure(figsize=(12, 6.5))
    sample_indices = [f"Sample {i+1}" for i in range(sample_size)]

    bars = plt.barh(sample_indices, jaccard_scores, color='steelblue', edgecolor='black', alpha=0.9, height=0.6)
    plt.axvline(mean_jaccard, color='crimson', linestyle='--', linewidth=2,
                label=f'Mean Agreement Bound ({mean_jaccard:.4f})')

    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.2f}',
                 va='center', ha='left', fontsize=9, fontweight='semibold', color='black')

    plt.xlabel('Top-5 Feature Jaccard Intersection Agreement Index', fontsize=11, fontweight='bold', labelpad=10)
    plt.ylabel('Evaluation Network Flow Index Slices', fontsize=11, fontweight='bold', labelpad=10)
    plt.title('Agnostic Attribution Space Integrity Consensus (True SHAP vs True Tabular LIME Explanations)',
              fontsize=12, fontweight='bold', pad=15)

    plt.xlim(0, 1.15)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='gray')
    plt.tight_layout()

    # --- FIXED PATH ROUTING ---
    # Safe fallback mapping using unified configuration structures
    try:
        drive_fig_path = os.path.join(CONFIG['results_dir'], 'shap_lime_jaccard_agreement.png')
        plt.savefig(drive_fig_path, dpi=300)
        print(f"[Visualization Saved] Plot synced to Drive: {drive_fig_path}")
    except Exception as e:
        # Fallback straight to root working directory if directory mappings break
        cwd_path = os.path.join(os.getcwd(), 'shap_lime_jaccard_agreement.png')
        plt.savefig(cwd_path, dpi=300)
        print(f"[Visualization Saved] Safe fallback path written to local workspace: {cwd_path}")

    plt.show()
    return mean_jaccard

# Pass variables directly inside your block pipeline cell execution sequence
global_xai_agreement = run_explanation_cross_validation(lgb_model, X_val, expected_features, train_df)


In [ ]:
# BLOCK 10: STAGE 6 — VALIDATION-BASED POLICY OPTIMIZATION
# Adds a validation-selected high-confidence benign override.
#
# IMPORTANT:
# 1. All six thresholds are selected ONLY on validation data.
# 2. All six thresholds must be frozen and reused unchanged downstream.
# 3. Your downstream routing/evaluation function must use
#    STRICT_OVERRIDE_PROB_TAU and STRICT_OVERRIDE_SIM_FLOOR as shown
#    in the routing rule at the bottom of this file.

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ANMR_SAFETY_FLOOR = float(CONFIG.get("anmr_safety_floor", 0.9999))
FBR_SAFETY_CEILING = float(CONFIG.get("fbr_safety_ceiling", 0.005))


def _safe_divide(numerator, denominator):
    return float(numerator) / max(float(denominator), 1.0)


def run_validation_policy_optimization(
    model,
    idx_calibrators,
    reference_signatures,
    X_val_mat,
    y_val_mat,
):
    """
    Select six operating thresholds once from validation data:

        1. TAU_P
        2. TAU_S
        3. BENIGN_PROB_TAU
        4. BENIGN_SIM_TAU
        5. OVERRIDE_PROB_TAU
        6. OVERRIDE_SIM_FLOOR

    The high-confidence benign override allows a predicted-benign sample
    to bypass the ordinary benign-similarity requirement only when:

        benign_probability >= OVERRIDE_PROB_TAU
        AND
        predicted_class_similarity >= OVERRIDE_SIM_FLOOR

    Hard feasibility constraints:
        ANMR >= ANMR_SAFETY_FLOOR
        FBR  <= FBR_SAFETY_CEILING

    Lexicographic objective among feasible policies:
        1. Minimize total escalation rate.
        2. Minimize false-block rate.
        3. Maximize ANMR.
        4. Prefer stricter override probability threshold.
        5. Prefer stricter override similarity floor.
        6. Prefer stricter ordinary benign probability threshold.
        7. Prefer stricter ordinary benign similarity threshold.

    The selected thresholds are frozen for every downstream test pillar.
    """

    print("=" * 115)
    print("[Stage 6] Validation-Based Six-Threshold Policy Optimization")
    print("=" * 115)
    sys.stdout.flush()

    X_val_mat = np.asarray(X_val_mat, dtype=np.float32)
    y_val_mat = np.asarray(y_val_mat)

    print("[Policy Optimization] Computing calibrated validation probabilities...")
    probabilities = get_calibrated_probabilities(
        model,
        idx_calibrators,
        X_val_mat,
    )

    predictions = np.argmax(probabilities, axis=1)
    benign_probability = probabilities[:, BENIGN_IDX]
    attack_confidence = 1.0 - benign_probability

    predicted_names = np.asarray(
        [resolve_active_class_name(index) for index in predictions],
        dtype=object,
    )

    true_names = np.asarray(
        [resolve_active_class_name(index) for index in y_val_mat],
        dtype=object,
    )

    predicted_benign = predicted_names == "Benign"
    true_benign = true_names == "Benign"
    true_malicious = ~true_benign

    n_flows = len(y_val_mat)
    total_benign = int(np.sum(true_benign))
    total_malicious = int(np.sum(true_malicious))

    print(
        "[Policy Optimization] Computing predicted-class explanation "
        "similarities with batched native LightGBM contributions..."
    )

    similarities, _ = compute_predicted_signature_similarity(
        model=model,
        reference_signatures=reference_signatures,
        X_matrix=X_val_mat,
        predicted_indices=predictions,
        predicted_names=predicted_names,
        batch_size=int(CONFIG.get("attribution_batch_size", 8192)),
    )

    # ------------------------------------------------------------------
    # SEARCH GRIDS
    # ------------------------------------------------------------------
    # Original four policy coordinates.
    tau_p_grid = np.linspace(0.60, 0.98, 16)
    tau_s_grid = np.linspace(0.01, 0.40, 16)
    benign_probability_grid = np.linspace(0.90, 0.99, 10)
    benign_similarity_grid = np.linspace(0.05, 0.40, 10)

    # New high-confidence benign override coordinates.
    #
    # These ranges are intentionally conservative:
    # - Override requires >= 0.995 benign probability.
    # - Similarity floor is never allowed below 0.00 in this search.
    #
    # This can recover many high-confidence benign flows whose explanation
    # similarity drifts below the ordinary 0.05+ release threshold while
    # avoiding an unrestricted similarity bypass.
    override_probability_grid = np.asarray(
        [0.9950, 0.9975, 0.9990, 0.9995],
        dtype=float,
    )

    override_similarity_floor_grid = np.asarray(
        [0.00, 0.01, 0.02, 0.03, 0.04],
        dtype=float,
    )

    print(
        "[Policy Optimization] Precomputing blocking masks and "
        "validation policy components..."
    )

    # ------------------------------------------------------------------
    # BLOCK CACHE
    # ------------------------------------------------------------------
    block_cache = {}

    for tau_p in tau_p_grid:
        confidence_mask = (
            (~predicted_benign)
            & (attack_confidence >= float(tau_p))
        )

        for tau_s in tau_s_grid:
            block_mask = (
                confidence_mask
                & (similarities >= (0.5 * float(tau_s)))
            )

            block_cache[(float(tau_p), float(tau_s))] = {
                "mask": block_mask,
                "count": int(np.sum(block_mask)),
                "false_blocks": int(np.sum(true_benign & block_mask)),
            }

    # ------------------------------------------------------------------
    # ALLOW CACHE
    # ------------------------------------------------------------------
    # We precompute the union of:
    #
    #   ordinary_allow OR high_confidence_override
    #
    # This avoids repeatedly constructing full Boolean masks in the
    # six-dimensional policy search.
    allow_cache = {}

    for tau_s in tau_s_grid:
        for tau_b in benign_probability_grid:
            ordinary_probability_mask = (
                predicted_benign
                & (benign_probability >= float(tau_b))
            )

            for tau_bs in benign_similarity_grid:
                effective_similarity = max(
                    float(tau_s),
                    float(tau_bs),
                )

                ordinary_allow = (
                    ordinary_probability_mask
                    & (similarities >= effective_similarity)
                )

                for override_prob_tau in override_probability_grid:
                    override_probability_mask = (
                        predicted_benign
                        & (
                            benign_probability
                            >= float(override_prob_tau)
                        )
                    )

                    for override_sim_floor in override_similarity_floor_grid:
                        high_confidence_override = (
                            override_probability_mask
                            & (
                                similarities
                                >= float(override_sim_floor)
                            )
                        )

                        # The override supplements the ordinary release rule.
                        # It does not affect predicted-malicious samples.
                        allow_mask = (
                            ordinary_allow
                            | high_confidence_override
                        )

                        key = (
                            float(tau_s),
                            float(tau_b),
                            float(tau_bs),
                            float(override_prob_tau),
                            float(override_sim_floor),
                        )

                        allow_cache[key] = {
                            "count": int(np.sum(allow_mask)),
                            "true_benign_allowed": int(
                                np.sum(true_benign & allow_mask)
                            ),
                            "missed_attacks": int(
                                np.sum(true_malicious & allow_mask)
                            ),
                        }

    feasible_rows = []
    best_thresholds = None
    best_selection_key = None

    print(
        "[Policy Optimization] Executing exact validation search "
        "with high-confidence benign override..."
    )
    sys.stdout.flush()

    # ------------------------------------------------------------------
    # EXACT VALIDATION SEARCH
    # ------------------------------------------------------------------
    for tau_p in tau_p_grid:
        for tau_s in tau_s_grid:
            block_stats = block_cache[
                (float(tau_p), float(tau_s))
            ]

            false_blocks = block_stats["false_blocks"]
            fbr = _safe_divide(
                false_blocks,
                total_benign,
            )

            # FBR is independent of benign-release thresholds.
            if fbr > FBR_SAFETY_CEILING:
                continue

            for tau_b in benign_probability_grid:
                for tau_bs in benign_similarity_grid:
                    for override_prob_tau in override_probability_grid:
                        for override_sim_floor in override_similarity_floor_grid:

                            allow_stats = allow_cache[
                                (
                                    float(tau_s),
                                    float(tau_b),
                                    float(tau_bs),
                                    float(override_prob_tau),
                                    float(override_sim_floor),
                                )
                            ]

                            missed_attacks = allow_stats[
                                "missed_attacks"
                            ]

                            anmr = 1.0 - _safe_divide(
                                missed_attacks,
                                total_malicious,
                            )

                            if anmr < ANMR_SAFETY_FLOOR:
                                continue

                            escalation_count = (
                                n_flows
                                - allow_stats["count"]
                                - block_stats["count"]
                            )

                            # Sanity guard. Because predicted-benign releases
                            # and predicted-malicious blocks are disjoint,
                            # this should never be negative.
                            if escalation_count < 0:
                                raise RuntimeError(
                                    "Negative escalation count detected. "
                                    "Check policy mask overlap."
                                )

                            escalation_rate = _safe_divide(
                                escalation_count,
                                n_flows,
                            )

                            benign_escalation_count = (
                                total_benign
                                - allow_stats["true_benign_allowed"]
                                - false_blocks
                            )

                            false_alarm_rate = _safe_divide(
                                benign_escalation_count,
                                total_benign,
                            )

                            row = {
                                "Tau_P": float(tau_p),
                                "Tau_S": float(tau_s),
                                "Benign_Prob_Tau": float(tau_b),
                                "Benign_Sim_Tau": float(tau_bs),
                                "Override_Prob_Tau": float(
                                    override_prob_tau
                                ),
                                "Override_Sim_Floor": float(
                                    override_sim_floor
                                ),
                                "ANMR": float(anmr),
                                "FBR": float(fbr),
                                "False_Alarm_Rate": float(
                                    false_alarm_rate
                                ),
                                "Escalation_Rate": float(
                                    escalation_rate
                                ),
                                "Missed_Attacks": int(
                                    missed_attacks
                                ),
                                "False_Blocks": int(
                                    false_blocks
                                ),
                                "Escalated_Flows": int(
                                    escalation_count
                                ),
                            }

                            feasible_rows.append(row)

                            # Lexicographic selection:
                            # 1. Minimize escalation.
                            # 2. Minimize FBR.
                            # 3. Maximize ANMR.
                            # 4. Prefer stricter override probability.
                            # 5. Prefer stricter override similarity floor.
                            # 6-9. Prefer stricter original thresholds.
                            selection_key = (
                                escalation_rate,
                                fbr,
                                -anmr,
                                -float(override_prob_tau),
                                -float(override_sim_floor),
                                -float(tau_b),
                                -float(tau_bs),
                                -float(tau_p),
                                -float(tau_s),
                            )

                            if (
                                best_selection_key is None
                                or selection_key < best_selection_key
                            ):
                                best_selection_key = selection_key

                                best_thresholds = (
                                    float(tau_p),
                                    float(tau_s),
                                    float(tau_b),
                                    float(tau_bs),
                                    float(override_prob_tau),
                                    float(override_sim_floor),
                                )

    if best_thresholds is None:
        raise RuntimeError(
            "No validation policy satisfied "
            f"ANMR >= {ANMR_SAFETY_FLOOR:.4f} and "
            f"FBR <= {FBR_SAFETY_CEILING:.4f}. "
            "Do not silently substitute default thresholds."
        )

    results_df = pd.DataFrame(
        feasible_rows
    ).sort_values(
        [
            "Escalation_Rate",
            "FBR",
            "ANMR",
        ],
        ascending=[
            True,
            True,
            False,
        ],
        ignore_index=True,
    )

    (
        best_tau_p,
        best_tau_s,
        best_tau_b,
        best_tau_bs,
        best_override_prob_tau,
        best_override_sim_floor,
    ) = best_thresholds

    best_row = results_df[
        (results_df["Tau_P"] == best_tau_p)
        & (results_df["Tau_S"] == best_tau_s)
        & (results_df["Benign_Prob_Tau"] == best_tau_b)
        & (results_df["Benign_Sim_Tau"] == best_tau_bs)
        & (
            results_df["Override_Prob_Tau"]
            == best_override_prob_tau
        )
        & (
            results_df["Override_Sim_Floor"]
            == best_override_sim_floor
        )
    ].iloc[0]

    print(
        "\n[Policy Optimization Complete] "
        "Feasible operating point selected."
    )

    print(f" -> TAU_P                    : {best_tau_p:.4f}")
    print(f" -> TAU_S                    : {best_tau_s:.4f}")
    print(f" -> BENIGN_PROB_TAU          : {best_tau_b:.4f}")
    print(f" -> BENIGN_SIM_TAU           : {best_tau_bs:.4f}")
    print(
        f" -> OVERRIDE_PROB_TAU        : "
        f"{best_override_prob_tau:.4f}"
    )
    print(
        f" -> OVERRIDE_SIM_FLOOR       : "
        f"{best_override_sim_floor:.4f}"
    )
    print(
        f" -> Validation ANMR          : "
        f"{best_row['ANMR']:.6f}"
    )
    print(
        f" -> Validation FBR           : "
        f"{best_row['FBR']:.6f}"
    )
    print(
        f" -> Validation escalation    : "
        f"{best_row['Escalation_Rate']:.6f}"
    )
    print(
        f" -> Validation false alarms  : "
        f"{best_row['False_Alarm_Rate']:.6f}"
    )
    print(
        f" -> Feasible policies        : "
        f"{len(results_df):,}"
    )
    sys.stdout.flush()

    # ------------------------------------------------------------------
    # VISUALIZATION
    # ------------------------------------------------------------------
    # Marginalize over all benign-release and override coordinates.
    reduced = (
        results_df.groupby(
            ["Tau_P", "Tau_S"],
            as_index=False,
        )["Escalation_Rate"]
        .min()
    )

    pivot = reduced.pivot(
        index="Tau_P",
        columns="Tau_S",
        values="Escalation_Rate",
    )

    plt.figure(figsize=(10, 8))

    image = plt.imshow(
        pivot.to_numpy(),
        aspect="auto",
        origin="lower",
    )

    plt.colorbar(
        image,
        label="Minimum Feasible Escalation Rate",
    )

    plt.xticks(
        np.arange(len(pivot.columns)),
        [
            f"{value:.3f}"
            for value in pivot.columns
        ],
        rotation=90,
    )

    plt.yticks(
        np.arange(len(pivot.index)),
        [
            f"{value:.3f}"
            for value in pivot.index
        ],
    )

    plt.title(
        "Validation-Based Six-Threshold Policy Optimization\n"
        "Minimum Feasible Escalation Rate"
    )

    plt.xlabel(
        r"Attack Similarity Threshold ($\tau_s$)"
    )

    plt.ylabel(
        r"Attack Confidence Threshold ($\tau_p$)"
    )

    plt.tight_layout()
    plt.show()

    validation_cache = {
        "probabilities": probabilities,
        "predictions": predictions,
        "predicted_names": predicted_names,
        "true_names": true_names,
        "attack_confidence": attack_confidence,
        "benign_probability": benign_probability,
        "similarities": similarities,
    }

    return (
        best_tau_p,
        best_tau_s,
        best_tau_b,
        best_tau_bs,
        best_override_prob_tau,
        best_override_sim_floor,
        results_df,
        validation_cache,
    )


(
    TAU_P,
    TAU_S,
    STRICT_BENIGN_PROB_TAU,
    STRICT_BENIGN_SIM_TAU,
    STRICT_OVERRIDE_PROB_TAU,
    STRICT_OVERRIDE_SIM_FLOOR,
    GRID_SEARCH_RESULTS_DF,
    PRIMARY_VALIDATION_CACHE,
) = run_validation_policy_optimization(
    lgb_model,
    calibrators,
    train_signatures,
    X_val,
    y_val,
)


TAU_P = float(TAU_P)
TAU_S = float(TAU_S)

STRICT_BENIGN_PROB_TAU = float(
    STRICT_BENIGN_PROB_TAU
)

STRICT_BENIGN_SIM_TAU = float(
    STRICT_BENIGN_SIM_TAU
)

STRICT_OVERRIDE_PROB_TAU = float(
    STRICT_OVERRIDE_PROB_TAU
)

STRICT_OVERRIDE_SIM_FLOOR = float(
    STRICT_OVERRIDE_SIM_FLOOR
)


print("\n" + "-" * 95)
print(
    "[Global Policy Lock] "
    "Validation-selected thresholds frozen downstream:"
)

print(
    f" -> TAU_P                    : "
    f"{TAU_P:.4f}"
)

print(
    f" -> TAU_S                    : "
    f"{TAU_S:.4f}"
)

print(
    f" -> BENIGN_PROB_TAU          : "
    f"{STRICT_BENIGN_PROB_TAU:.4f}"
)

print(
    f" -> BENIGN_SIM_TAU           : "
    f"{STRICT_BENIGN_SIM_TAU:.4f}"
)

print(
    f" -> OVERRIDE_PROB_TAU        : "
    f"{STRICT_OVERRIDE_PROB_TAU:.4f}"
)

print(
    f" -> OVERRIDE_SIM_FLOOR       : "
    f"{STRICT_OVERRIDE_SIM_FLOOR:.4f}"
)

print("-" * 95)
sys.stdout.flush()

### Validation-Optimized Selective Attribution

The original full-attribution routing policy is first established on validation data. This stage then learns which highly confident predictions may be resolved without SHAP while preserving the same validation ANMR/FBR safety constraints. Downstream test environments do not participate in this optimization.


In [ ]:

# BLOCK 10B: VALIDATION-OPTIMIZED SELECTIVE ATTRIBUTION SCREEN
#
# PURPOSE
# -------
# Reduce the number of expensive SHAP attribution computations without
# selecting bypass rules from any downstream test environment.
#
# The current 100%-SHAP policy remains the safety baseline.
# This block uses ONLY the validation cache created in Block 10.
#
# A flow can bypass SHAP only through one of two confidence screens:
#
#   1. Direct benign release:
#        predicted Benign
#        AND benign probability >= selected threshold
#        AND top1-vs-top2 probability margin >= selected threshold
#
#   2. Direct malicious block:
#        predicted non-Benign
#        AND attack confidence >= TAU_P
#        AND predicted-class probability >= selected threshold
#        AND top1-vs-top2 probability margin >= selected threshold
#
# Every remaining flow receives the original SHAP/cosine verification policy.
#
# Optimization objective among validation-feasible policies:
#   1. Minimize fraction of flows requiring attribution.
#   2. Minimize escalation.
#   3. Minimize FBR.
#   4. Maximize ANMR.
#
# Safety constraints remain identical to Block 10:
#   ANMR >= ANMR_SAFETY_FLOOR
#   FBR  <= FBR_SAFETY_CEILING

import numpy as np
import pandas as pd

print("=" * 115)
print("[Block 10B] Validation-Optimized Selective Attribution Screen")
print("=" * 115)

if "PRIMARY_VALIDATION_CACHE" not in globals():
    raise RuntimeError(
        "PRIMARY_VALIDATION_CACHE is unavailable. Run Block 10 first."
    )

cache = PRIMARY_VALIDATION_CACHE

required_cache_keys = {
    "probabilities",
    "predictions",
    "predicted_names",
    "true_names",
    "attack_confidence",
    "benign_probability",
    "similarities",
}
missing_cache_keys = required_cache_keys.difference(cache.keys())
if missing_cache_keys:
    raise RuntimeError(
        "Validation cache is missing required fields: "
        + ", ".join(sorted(missing_cache_keys))
    )

val_probabilities = np.asarray(cache["probabilities"], dtype=float)
val_predictions = np.asarray(cache["predictions"], dtype=int)
val_predicted_names = np.asarray(cache["predicted_names"], dtype=object)
val_true_names = np.asarray(cache["true_names"], dtype=object)
val_attack_confidence = np.asarray(cache["attack_confidence"], dtype=float)
val_benign_probability = np.asarray(cache["benign_probability"], dtype=float)
val_similarities = np.asarray(cache["similarities"], dtype=float)

if val_probabilities.ndim != 2:
    raise ValueError("Validation probabilities must be a 2-D matrix.")

# Top-1 probability and top-1/top-2 margin are inexpensive confidence signals.
val_top1_probability = np.max(val_probabilities, axis=1)
if val_probabilities.shape[1] > 1:
    top2_partition = np.partition(
        val_probabilities,
        kth=val_probabilities.shape[1] - 2,
        axis=1,
    )
    val_second_probability = top2_partition[:, -2]
else:
    val_second_probability = np.zeros(len(val_probabilities), dtype=float)

val_probability_margin = val_top1_probability - val_second_probability

val_true_benign = val_true_names == "Benign"
val_true_malicious = ~val_true_benign
val_predicted_benign = val_predicted_names == "Benign"

# Baseline 100%-SHAP routing is the reference behavior.
baseline_routing = route_policy_batch(
    pred_names=val_predicted_names,
    true_names=val_true_names,
    attack_confidence=val_attack_confidence,
    similarities=val_similarities,
    tau_p=TAU_P,
    tau_s=TAU_S,
    benign_prob_tau=STRICT_BENIGN_PROB_TAU,
    benign_sim_tau=STRICT_BENIGN_SIM_TAU,
    override_prob_tau=STRICT_OVERRIDE_PROB_TAU,
    override_sim_floor=STRICT_OVERRIDE_SIM_FLOOR,
)

n_val = len(val_true_names)
total_val_malicious = int(np.sum(val_true_malicious))
total_val_benign = int(np.sum(val_true_benign))

# 1.01 is an explicit "disabled" option because calibrated probabilities
# and margins cannot exceed 1.0.
benign_direct_prob_grid = np.asarray(
    CONFIG.get(
        "selective_benign_direct_prob_grid",
        [0.9950, 0.9975, 0.9990, 0.9995, 0.9999, 1.01],
    ),
    dtype=float,
)
benign_direct_margin_grid = np.asarray(
    CONFIG.get(
        "selective_benign_direct_margin_grid",
        [0.90, 0.95, 0.98, 0.99, 1.01],
    ),
    dtype=float,
)
attack_direct_prob_grid = np.asarray(
    CONFIG.get(
        "selective_attack_direct_prob_grid",
        [0.90, 0.95, 0.97, 0.99, 0.995, 1.01],
    ),
    dtype=float,
)
attack_direct_margin_grid = np.asarray(
    CONFIG.get(
        "selective_attack_direct_margin_grid",
        [0.80, 0.90, 0.95, 0.98, 0.99, 1.01],
    ),
    dtype=float,
)

selective_rows = []
best_key = None
best_row = None

for benign_prob_tau in benign_direct_prob_grid:
    for benign_margin_tau in benign_direct_margin_grid:

        direct_allow = (
            val_predicted_benign
            & (val_benign_probability >= float(benign_prob_tau))
            & (val_probability_margin >= float(benign_margin_tau))
        )

        for attack_prob_tau in attack_direct_prob_grid:
            for attack_margin_tau in attack_direct_margin_grid:

                direct_block = (
                    (~val_predicted_benign)
                    & (val_attack_confidence >= float(TAU_P))
                    & (val_top1_probability >= float(attack_prob_tau))
                    & (val_probability_margin >= float(attack_margin_tau))
                )

                # Predicted benign and predicted malicious screens are disjoint.
                verify_mask = ~(direct_allow | direct_block)

                # All non-bypassed rows retain the exact original SHAP policy.
                tier_1 = baseline_routing["tier_1_allow"].copy()
                tier_2 = baseline_routing["tier_2_block"].copy()

                # Confidence-only bypass decisions replace only unresolved
                # decisions for their respective prediction side.
                tier_1[direct_allow] = True
                tier_2[direct_allow] = False

                tier_2[direct_block] = True
                tier_1[direct_block] = False

                tier_3 = ~(tier_1 | tier_2)

                missed_attacks = int(
                    np.sum(val_true_malicious & tier_1)
                )
                false_blocks = int(
                    np.sum(val_true_benign & tier_2)
                )

                anmr = (
                    1.0
                    - missed_attacks / max(total_val_malicious, 1)
                )
                fbr = (
                    false_blocks / max(total_val_benign, 1)
                )

                if (
                    anmr < ANMR_SAFETY_FLOOR
                    or fbr > FBR_SAFETY_CEILING
                ):
                    continue

                escalation_rate = float(np.mean(tier_3))
                attribution_fraction = float(np.mean(verify_mask))
                direct_allow_rate = float(np.mean(direct_allow))
                direct_block_rate = float(np.mean(direct_block))

                row = {
                    "Benign_Direct_Prob_Tau": float(benign_prob_tau),
                    "Benign_Direct_Margin_Tau": float(benign_margin_tau),
                    "Attack_Direct_Prob_Tau": float(attack_prob_tau),
                    "Attack_Direct_Margin_Tau": float(attack_margin_tau),
                    "Validation_ANMR": float(anmr),
                    "Validation_FBR": float(fbr),
                    "Validation_Escalation_Rate": escalation_rate,
                    "Attribution_Evaluated_Fraction": attribution_fraction,
                    "Direct_Allow_Rate": direct_allow_rate,
                    "Direct_Block_Rate": direct_block_rate,
                }
                selective_rows.append(row)

                # Throughput-oriented lexicographic selection.
                selection_key = (
                    attribution_fraction,
                    escalation_rate,
                    fbr,
                    -anmr,
                    -float(benign_prob_tau),
                    -float(benign_margin_tau),
                    -float(attack_prob_tau),
                    -float(attack_margin_tau),
                )

                if best_key is None or selection_key < best_key:
                    best_key = selection_key
                    best_row = row

if best_row is None:
    raise RuntimeError(
        "No selective-attribution operating point satisfied the "
        "validation ANMR/FBR safety constraints."
    )

SELECTIVE_VALIDATION_RESULTS_DF = pd.DataFrame(selective_rows).sort_values(
    [
        "Attribution_Evaluated_Fraction",
        "Validation_Escalation_Rate",
        "Validation_FBR",
        "Validation_ANMR",
    ],
    ascending=[True, True, True, False],
    ignore_index=True,
)

SELECTIVE_VALIDATION_BEST_ROW = pd.Series(best_row)

SELECTIVE_BENIGN_DIRECT_PROB_TAU = float(
    best_row["Benign_Direct_Prob_Tau"]
)
SELECTIVE_BENIGN_DIRECT_MARGIN_TAU = float(
    best_row["Benign_Direct_Margin_Tau"]
)
SELECTIVE_ATTACK_DIRECT_PROB_TAU = float(
    best_row["Attack_Direct_Prob_Tau"]
)
SELECTIVE_ATTACK_DIRECT_MARGIN_TAU = float(
    best_row["Attack_Direct_Margin_Tau"]
)

print("[Selective Attribution] Validation-selected screen:")
print(
    f" -> BENIGN_DIRECT_PROB_TAU   : "
    f"{SELECTIVE_BENIGN_DIRECT_PROB_TAU:.4f}"
)
print(
    f" -> BENIGN_DIRECT_MARGIN_TAU : "
    f"{SELECTIVE_BENIGN_DIRECT_MARGIN_TAU:.4f}"
)
print(
    f" -> ATTACK_DIRECT_PROB_TAU   : "
    f"{SELECTIVE_ATTACK_DIRECT_PROB_TAU:.4f}"
)
print(
    f" -> ATTACK_DIRECT_MARGIN_TAU : "
    f"{SELECTIVE_ATTACK_DIRECT_MARGIN_TAU:.4f}"
)
print(
    f" -> Validation ANMR          : "
    f"{best_row['Validation_ANMR']:.6f}"
)
print(
    f" -> Validation FBR           : "
    f"{best_row['Validation_FBR']:.6f}"
)
print(
    f" -> Validation escalation    : "
    f"{best_row['Validation_Escalation_Rate']:.6f}"
)
print(
    f" -> SHAP evaluated fraction  : "
    f"{best_row['Attribution_Evaluated_Fraction']:.6f}"
)
print(
    f" -> Direct allow rate        : "
    f"{best_row['Direct_Allow_Rate']:.6f}"
)
print(
    f" -> Direct block rate        : "
    f"{best_row['Direct_Block_Rate']:.6f}"
)
print("=" * 115)


In [ ]:
# BLOCK 11: THREE-TIER EXPLANATION-VERIFIED OPERATIONAL EVALUATOR

import os
import time
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import accuracy_score, f1_score

print("=" * 115)
print("[System Core] Compiling the harmonized three-tier evaluator...")
print("=" * 115)


def _estimate_packet_count_from_matrix(X_matrix, feature_names):
    """Estimate represented packet volume from forward/backward packet counts."""
    X_matrix = np.asarray(X_matrix)
    name_to_index = {
        str(name): index for index, name in enumerate(feature_names)
    }

    if {
        "fwd_packets_count",
        "bwd_packets_count",
    }.issubset(name_to_index):
        forward = np.nan_to_num(
            X_matrix[:, name_to_index["fwd_packets_count"]].astype(float),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        backward = np.nan_to_num(
            X_matrix[:, name_to_index["bwd_packets_count"]].astype(float),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        represented_packets = float(
            np.sum(np.maximum(forward, 0.0) + np.maximum(backward, 0.0))
        )
        return max(represented_packets, float(len(X_matrix)))

    return float(len(X_matrix))


def evaluate_operational_environment(
    model,
    idx_calibrators,
    reference_signatures,
    tp,
    ts,
    X_matrix,
    y_matrix,
    environment_tag,
    benign_prob_tau=None,
    benign_sim_tau=None,
    probability_fn=None,
    return_decisions=False,
    attribution_batch_size=None,
):
    """Evaluate one environment with the frozen three-tier policy."""
    X_matrix = np.asarray(X_matrix, dtype=np.float32)
    y_matrix = np.asarray(y_matrix)

    if len(X_matrix) == 0:
        raise ValueError(f"{environment_tag} contains no evaluation flows.")
    if len(X_matrix) != len(y_matrix):
        raise ValueError("X_matrix and y_matrix must contain equal rows.")

    probability_fn = probability_fn or get_calibrated_probabilities
    attribution_batch_size = int(
        attribution_batch_size
        or CONFIG.get("attribution_batch_size", 8192)
    )
    benign_prob_tau = float(
        STRICT_BENIGN_PROB_TAU
        if benign_prob_tau is None
        else benign_prob_tau
    )
    benign_sim_tau = float(
        STRICT_BENIGN_SIM_TAU
        if benign_sim_tau is None
        else benign_sim_tau
    )

    process = psutil.Process(os.getpid())
    memory_baseline = process.memory_info().rss / (1024 * 1024)
    n_flows = len(X_matrix)

    print(f"\n[Three-Tier Evaluation] {environment_tag}")
    total_start = time.perf_counter()

    probability_start = time.perf_counter()
    probabilities = probability_fn(model, idx_calibrators, X_matrix)
    probability_elapsed = time.perf_counter() - probability_start

    predictions = np.argmax(probabilities, axis=1)
    predicted_names = np.asarray(
        [resolve_active_class_name(index) for index in predictions],
        dtype=object,
    )
    true_names = np.asarray(
        [resolve_active_class_name(index) for index in y_matrix],
        dtype=object,
    )
    benign_probability = probabilities[:, BENIGN_IDX]
    attack_confidence = 1.0 - benign_probability

    attribution_start = time.perf_counter()
    similarities, _ = compute_predicted_signature_similarity(
        model=model,
        reference_signatures=reference_signatures,
        X_matrix=X_matrix,
        predicted_indices=predictions,
        predicted_names=predicted_names,
        batch_size=attribution_batch_size,
    )
    attribution_elapsed = time.perf_counter() - attribution_start
    routing = route_policy_batch(
        pred_names=predicted_names,
        true_names=true_names,
        attack_confidence=attack_confidence,
        similarities=similarities,
        tau_p=tp,
        tau_s=ts,
        benign_prob_tau=benign_prob_tau,
        benign_sim_tau=benign_sim_tau,
        override_prob_tau=STRICT_OVERRIDE_PROB_TAU,
        override_sim_floor=STRICT_OVERRIDE_SIM_FLOOR,
    )
    end_to_end_elapsed = time.perf_counter() - total_start

    true_benign = true_names == "Benign"
    true_malicious = ~true_benign
    total_malicious = int(np.sum(true_malicious))
    total_benign = int(np.sum(true_benign))
    missed_attacks = int(np.sum(routing["missed_attack"]))
    false_blocks = int(np.sum(routing["false_block"]))
    benign_escalations = int(np.sum(routing["false_alarm"]))
    escalation_count = int(np.sum(routing["tier_3_escalate"]))

    known_mask = y_matrix != ZERO_DAY_ENCODED_LABEL
    if np.any(known_mask):
        global_accuracy = accuracy_score(
            y_matrix[known_mask], predictions[known_mask]
        )
        macro_f1 = f1_score(
            y_matrix[known_mask],
            predictions[known_mask],
            average="macro",
            zero_division=0,
        )
    else:
        global_accuracy = np.nan
        macro_f1 = np.nan

    total_packets = _estimate_packet_count_from_matrix(
        X_matrix, expected_features
    )
    memory_terminal = process.memory_info().rss / (1024 * 1024)

    result = {
        "Target_Environment": environment_tag,
        "Label_Diversity_Score": len(np.unique(y_matrix)),
        "Global_Accuracy": float(global_accuracy),
        "Macro_F1_Throughput": float(macro_f1),
        "Attack_Not_Missed_Rate_ANMR":
            1.0 - missed_attacks / max(total_malicious, 1),
        "False_Block_Rate_FBR":
            false_blocks / max(total_benign, 1),
        "False_Alarm_Escalation_Rate":
            benign_escalations / max(total_benign, 1),
        "Human_Escalation_Rate": escalation_count / n_flows,
        "Escalation_Rate": escalation_count / n_flows,
        "Probability_Latency_ms_per_flow":
            1000.0 * probability_elapsed / n_flows,
        "Attribution_Latency_ms_per_flow":
            1000.0 * attribution_elapsed / n_flows,
        "End_to_End_Latency_ms_per_flow":
            1000.0 * end_to_end_elapsed / n_flows,
        "Latency_ms_per_packet":
            1000.0 * end_to_end_elapsed / max(total_packets, 1.0),
        "Primary_Throughput_flows_per_sec":
            n_flows / max(probability_elapsed, 1e-12),
        "End_to_End_Throughput_flows_per_sec":
            n_flows / max(end_to_end_elapsed, 1e-12),
        "Throughput_packets_per_sec":
            total_packets / max(end_to_end_elapsed, 1e-12),
        "Estimated_Packets_Represented": total_packets,
        "RAM_Footprint_Overhead_MB":
            max(0.0, memory_terminal - memory_baseline),
        "Tier_1_ALLOW": int(np.sum(routing["tier_1_allow"])),
        "Tier_2_AUTONOMOUS_BLOCK": int(np.sum(routing["tier_2_block"])),
        "Tier_3_ADMINISTRATIVE_ESCALATION": escalation_count,
        "Silent_Breaches": missed_attacks,
    }

    if return_decisions:
        result["_Decision_Log"] = pd.DataFrame({
            "Predicted_Class": predicted_names,
            "True_Class": true_names,
            "Benign_Probability": benign_probability,
            "Attack_Confidence": attack_confidence,
            "Predicted_Class_Similarity": similarities,

            "Effective_Benign_Similarity_Threshold":
                routing[
                    "effective_benign_similarity_threshold"
                ],

            "Malicious_Similarity_Threshold":
                routing[
                    "malicious_similarity_threshold"
                ],

            "Override_Probability_Threshold":
                routing[
                    "override_probability_threshold"
                ],

            "Override_Similarity_Floor":
                routing[
                    "override_similarity_floor"
                ],

            "Ordinary_Benign_Allow":
                routing[
                    "ordinary_benign_allow"
                ],

            "High_Confidence_Benign_Override":
                routing[
                    "high_confidence_benign_override"
                ],

            "Action": routing["actions"],
            "Tier": routing["tiers"],
            "Missed_Attack": routing["missed_attack"],
            "False_Block": routing["false_block"],
            "Benign_False_Escalation":
                routing["false_alarm"],
        })
    return result


print(
    "[System Core] Compiled: Tier 1 Allow, Tier 2 Autonomous Block, "
    "Tier 3 Administrative Escalation."
)


In [ ]:

# BLOCK 11B: SELECTIVE-ATTRIBUTION THREE-TIER EVALUATOR
#
# This evaluator is intentionally separate from the 100%-SHAP evaluator in
# Block 11 so that the manuscript can compare both policies fairly.
#
# Only validation-selected confidence-screened flows bypass SHAP.
# Every other flow uses the exact frozen explanation-guided routing policy.

import os
import time
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import accuracy_score, f1_score

print("=" * 115)
print("[System Core] Compiling selective-attribution evaluator...")
print("=" * 115)


def evaluate_selective_operational_environment(
    model,
    idx_calibrators,
    reference_signatures,
    tp,
    ts,
    X_matrix,
    y_matrix,
    environment_tag,
    benign_prob_tau=None,
    benign_sim_tau=None,
    probability_fn=None,
    return_decisions=False,
    attribution_batch_size=None,
):
    """Evaluate one environment with validation-frozen selective SHAP."""

    required_thresholds = [
        "SELECTIVE_BENIGN_DIRECT_PROB_TAU",
        "SELECTIVE_BENIGN_DIRECT_MARGIN_TAU",
        "SELECTIVE_ATTACK_DIRECT_PROB_TAU",
        "SELECTIVE_ATTACK_DIRECT_MARGIN_TAU",
    ]
    missing = [
        name for name in required_thresholds
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            "Run Block 10B before selective evaluation. Missing: "
            + ", ".join(missing)
        )

    X_matrix = np.asarray(X_matrix, dtype=np.float32)
    y_matrix = np.asarray(y_matrix)

    if len(X_matrix) == 0:
        raise ValueError(f"{environment_tag} contains no evaluation flows.")
    if len(X_matrix) != len(y_matrix):
        raise ValueError("X_matrix and y_matrix must contain equal rows.")

    probability_fn = probability_fn or get_calibrated_probabilities
    attribution_batch_size = int(
        attribution_batch_size
        or CONFIG.get("attribution_batch_size", 8192)
    )
    benign_prob_tau = float(
        STRICT_BENIGN_PROB_TAU
        if benign_prob_tau is None
        else benign_prob_tau
    )
    benign_sim_tau = float(
        STRICT_BENIGN_SIM_TAU
        if benign_sim_tau is None
        else benign_sim_tau
    )

    process = psutil.Process(os.getpid())
    memory_baseline = process.memory_info().rss / (1024 * 1024)
    n_flows = len(X_matrix)

    print(f"\n[Selective Evaluation] {environment_tag}")
    total_start = time.perf_counter()

    # --------------------------------------------------------------
    # Stage 1: calibrated probability inference for every flow.
    # --------------------------------------------------------------
    probability_start = time.perf_counter()
    probabilities = probability_fn(
        model,
        idx_calibrators,
        X_matrix,
    )
    probability_elapsed = time.perf_counter() - probability_start

    predictions = np.argmax(probabilities, axis=1)
    predicted_names = np.asarray(
        [resolve_active_class_name(index) for index in predictions],
        dtype=object,
    )
    true_names = np.asarray(
        [resolve_active_class_name(index) for index in y_matrix],
        dtype=object,
    )

    benign_probability = probabilities[:, BENIGN_IDX]
    attack_confidence = 1.0 - benign_probability

    top1_probability = np.max(probabilities, axis=1)
    if probabilities.shape[1] > 1:
        partitioned = np.partition(
            probabilities,
            kth=probabilities.shape[1] - 2,
            axis=1,
        )
        second_probability = partitioned[:, -2]
    else:
        second_probability = np.zeros(n_flows, dtype=float)

    probability_margin = top1_probability - second_probability
    predicted_benign = predicted_names == "Benign"

    # --------------------------------------------------------------
    # Stage 2: validation-selected confidence-only bypass.
    # --------------------------------------------------------------
    direct_allow = (
        predicted_benign
        & (
            benign_probability
            >= SELECTIVE_BENIGN_DIRECT_PROB_TAU
        )
        & (
            probability_margin
            >= SELECTIVE_BENIGN_DIRECT_MARGIN_TAU
        )
    )

    direct_block = (
        (~predicted_benign)
        & (attack_confidence >= float(tp))
        & (
            top1_probability
            >= SELECTIVE_ATTACK_DIRECT_PROB_TAU
        )
        & (
            probability_margin
            >= SELECTIVE_ATTACK_DIRECT_MARGIN_TAU
        )
    )

    verify_mask = ~(direct_allow | direct_block)
    verify_indices = np.where(verify_mask)[0]

    # --------------------------------------------------------------
    # Stage 3: SHAP/cosine only for unresolved flows.
    # --------------------------------------------------------------
    similarities = np.full(
        n_flows,
        np.nan,
        dtype=np.float32,
    )

    actions = np.full(
        n_flows,
        "ESCALATE",
        dtype=object,
    )
    tiers = np.full(
        n_flows,
        "Tier 3",
        dtype=object,
    )

    tier_1_allow = np.zeros(n_flows, dtype=bool)
    tier_2_block = np.zeros(n_flows, dtype=bool)
    tier_3_escalate = np.ones(n_flows, dtype=bool)

    ordinary_benign_allow = np.zeros(n_flows, dtype=bool)
    high_confidence_benign_override = np.zeros(n_flows, dtype=bool)

    # Confidence-only bypass decisions.
    tier_1_allow[direct_allow] = True
    tier_3_escalate[direct_allow] = False
    actions[direct_allow] = "ALLOW"
    tiers[direct_allow] = "Tier 1"

    tier_2_block[direct_block] = True
    tier_3_escalate[direct_block] = False
    actions[direct_block] = "BLOCK"
    tiers[direct_block] = "Tier 2"

    attribution_start = time.perf_counter()

    if len(verify_indices) > 0:
        verify_similarities, _ = compute_predicted_signature_similarity(
            model=model,
            reference_signatures=reference_signatures,
            X_matrix=X_matrix[verify_indices],
            predicted_indices=predictions[verify_indices],
            predicted_names=predicted_names[verify_indices],
            batch_size=attribution_batch_size,
        )
        similarities[verify_indices] = verify_similarities

        verified_routing = route_policy_batch(
            pred_names=predicted_names[verify_indices],
            true_names=true_names[verify_indices],
            attack_confidence=attack_confidence[verify_indices],
            similarities=verify_similarities,
            tau_p=tp,
            tau_s=ts,
            benign_prob_tau=benign_prob_tau,
            benign_sim_tau=benign_sim_tau,
            override_prob_tau=STRICT_OVERRIDE_PROB_TAU,
            override_sim_floor=STRICT_OVERRIDE_SIM_FLOOR,
        )

        actions[verify_indices] = verified_routing["actions"]
        tiers[verify_indices] = verified_routing["tiers"]

        tier_1_allow[verify_indices] = verified_routing["tier_1_allow"]
        tier_2_block[verify_indices] = verified_routing["tier_2_block"]
        tier_3_escalate[verify_indices] = verified_routing["tier_3_escalate"]

        ordinary_benign_allow[verify_indices] = (
            verified_routing["ordinary_benign_allow"]
        )
        high_confidence_benign_override[verify_indices] = (
            verified_routing["high_confidence_benign_override"]
        )

    attribution_elapsed = time.perf_counter() - attribution_start
    end_to_end_elapsed = time.perf_counter() - total_start

    # --------------------------------------------------------------
    # Metrics use the same definitions as the 100%-SHAP evaluator.
    # --------------------------------------------------------------
    true_benign = true_names == "Benign"
    true_malicious = ~true_benign

    missed_attack = true_malicious & tier_1_allow
    false_block = true_benign & tier_2_block
    false_alarm = true_benign & tier_3_escalate

    total_malicious = int(np.sum(true_malicious))
    total_benign = int(np.sum(true_benign))
    missed_attacks = int(np.sum(missed_attack))
    false_blocks = int(np.sum(false_block))
    escalation_count = int(np.sum(tier_3_escalate))

    known_mask = y_matrix != ZERO_DAY_ENCODED_LABEL
    if np.any(known_mask):
        global_accuracy = accuracy_score(
            y_matrix[known_mask],
            predictions[known_mask],
        )
        macro_f1 = f1_score(
            y_matrix[known_mask],
            predictions[known_mask],
            average="macro",
            zero_division=0,
        )
    else:
        global_accuracy = np.nan
        macro_f1 = np.nan

    memory_terminal = process.memory_info().rss / (1024 * 1024)

    attribution_fraction = len(verify_indices) / max(n_flows, 1)
    direct_allow_rate = float(np.mean(direct_allow))
    direct_block_rate = float(np.mean(direct_block))

    result = {
        "Target_Environment": environment_tag,
        "Policy_Mode": "Selective Attribution",
        "Label_Diversity_Score": len(np.unique(y_matrix)),
        "Global_Accuracy": float(global_accuracy),
        "Macro_F1_Throughput": float(macro_f1),
        "Attack_Not_Missed_Rate_ANMR":
            1.0 - missed_attacks / max(total_malicious, 1),
        "False_Block_Rate_FBR":
            false_blocks / max(total_benign, 1),
        "Escalation_Rate":
            escalation_count / n_flows,
        "Human_Escalation_Rate":
            escalation_count / n_flows,
        "False_Alarm_Escalation_Rate":
            int(np.sum(false_alarm)) / max(total_benign, 1),
        "Probability_Latency_ms_per_flow":
            1000.0 * probability_elapsed / n_flows,
        # Amortized attribution cost over all input flows.
        "Attribution_Latency_ms_per_flow":
            1000.0 * attribution_elapsed / n_flows,
        # Diagnostic: actual SHAP cost among only the verified rows.
        "Attribution_Latency_ms_per_verified_flow":
            (
                1000.0 * attribution_elapsed / len(verify_indices)
                if len(verify_indices) > 0
                else 0.0
            ),
        "End_to_End_Latency_ms_per_flow":
            1000.0 * end_to_end_elapsed / n_flows,
        "End_to_End_Throughput_flows_per_sec":
            n_flows / max(end_to_end_elapsed, 1e-12),
        "RAM_Footprint_Overhead_MB":
            max(0.0, memory_terminal - memory_baseline),
        "Attribution_Evaluated_Fraction":
            float(attribution_fraction),
        "Direct_Confidence_Allow_Rate":
            direct_allow_rate,
        "Direct_Confidence_Block_Rate":
            direct_block_rate,
        "Tier_1_ALLOW": int(np.sum(tier_1_allow)),
        "Tier_2_AUTONOMOUS_BLOCK": int(np.sum(tier_2_block)),
        "Tier_3_ADMINISTRATIVE_ESCALATION":
            int(np.sum(tier_3_escalate)),
        "Silent_Breaches": missed_attacks,
    }

    if return_decisions:
        result["_Decision_Log"] = pd.DataFrame({
            "Predicted_Class": predicted_names,
            "True_Class": true_names,
            "Benign_Probability": benign_probability,
            "Top1_Probability": top1_probability,
            "Probability_Margin": probability_margin,
            "Attribution_Evaluated": verify_mask,
            "Predicted_Class_Similarity": similarities,
            "Direct_Confidence_Allow": direct_allow,
            "Direct_Confidence_Block": direct_block,
            "Action": actions,
            "Tier": tiers,
            "Missed_Attack": missed_attack,
            "False_Block": false_block,
            "Benign_False_Escalation": false_alarm,
        })

    return result


print(
    "[System Core] Selective evaluator compiled. "
    "100%-SHAP Block 11 remains unchanged for baseline comparison."
)


In [ ]:

# BLOCK 12: PILLARS 1–3 — NATURAL CONCEPT DRIFT WITH SELECTIVE ATTRIBUTION
import pandas as pd
import sys

print("=" * 115)
print("[Pillars 1–3] Selective-attribution evaluation across natural concept drift...")
print("=" * 115)
sys.stdout.flush()

if "partition_caches" in globals() or "partition_caches" in locals():
    _, _, host_test_df, time_test_df, hard_test_df = partition_caches[0.7]
else:
    raise NameError(
        "partition_caches is unavailable. Run the partitioning stage first."
    )

drift_environments_list = [
    ("Host-Test (Topological Shift)", host_test_df),
    ("Time-Test (Temporal Drift)", time_test_df),
    ("Hard-Test (Combined Shift)", hard_test_df),
]

if (
    "comprehensive_metrics_log" not in globals()
    and "comprehensive_metrics_log" not in locals()
):
    comprehensive_metrics_log = []

block_12_records = []

for tag, df_slice in drift_environments_list:
    X_test_env = (
        df_slice[expected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    y_test_env = encode_active_labels(
        df_slice["label"],
        zero_day_value=None,
    )

    metrics = evaluate_selective_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_test_env,
        y_matrix=y_test_env,
        environment_tag=tag,
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        probability_fn=get_calibrated_probabilities,
        return_decisions=False,
    )

    comprehensive_metrics_log.append(metrics)
    block_12_records.append(metrics)

block_12_df = pd.DataFrame(block_12_records)

display_columns = [
    "Target_Environment",
    "Global_Accuracy",
    "Macro_F1_Throughput",
    "Attack_Not_Missed_Rate_ANMR",
    "False_Block_Rate_FBR",
    "Escalation_Rate",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
    "RAM_Footprint_Overhead_MB",
]

print("\n" + "#" * 115)
print(" PILLARS 1–3: SELECTIVE-ATTRIBUTION NATURAL DRIFT RESULTS ")
print("#" * 115)
print(
    block_12_df[
        [c for c in display_columns if c in block_12_df.columns]
    ].to_string(index=False)
)
print("#" * 115)
sys.stdout.flush()


In [ ]:
# DIAGNOSTIC SUB-CELL: NATURAL DRIFT POLICY AUDIT — SHARED THREE-TIER POLICY

import pandas as pd
import sys

print("=" * 115)
print(" SELECTIVE-ATTRIBUTION NATURAL CONCEPT DRIFT ROUTING AUDIT ")
print("=" * 115)

_, _, host_test_df, time_test_df, hard_test_df = partition_caches[0.7]

all_drift_environments = [
    (
        "Host-Test (Topological Shift Only)",
        host_test_df,
    ),
    (
        "Time-Test (Temporal Drift Only)",
        time_test_df,
    ),
    (
        "Hard-Test (Combined Spatial-Temporal Shift)",
        hard_test_df,
    ),
]

records = []

for env_name, df_slice in all_drift_environments:

    X_diag = df_slice[
        expected_features
    ].to_numpy()

    y_diag = encode_active_labels(
        df_slice["label"],
        zero_day_value=None,
    )

    result = evaluate_selective_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_diag,
        y_matrix=y_diag,
        environment_tag=env_name,
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        return_decisions=True,
    )

    decision_log = result.pop(
        "_Decision_Log"
    )

    records.append({
        "Environment":
            env_name,

        "Tier 1 (Allow)":
            int(
                (
                    decision_log["Tier"]
                    == "Tier 1"
                ).sum()
            ),

        "Tier 2 (Autonomous Block)":
            int(
                (
                    decision_log["Tier"]
                    == "Tier 2"
                ).sum()
            ),

        "Tier 3 (Administrative Escalation)":
            int(
                (
                    decision_log["Tier"]
                    == "Tier 3"
                ).sum()
            ),

        "ANMR":
            result[
                "Attack_Not_Missed_Rate_ANMR"
            ],

        "FBR":
            result[
                "False_Block_Rate_FBR"
            ],

        "Escalation":
            result[
                "Escalation_Rate"
            ]
    })

drift_policy_audit_df = pd.DataFrame(
    records
)

print(
    drift_policy_audit_df.to_string(
        index=False
    )
)

sys.stdout.flush()


In [ ]:

# BLOCK 13: PILLAR 4 — ZERO-DAY ATTACK HOLDOUT WITH SELECTIVE ATTRIBUTION
import numpy as np
import pandas as pd

print("=" * 115)
print(
    f"[Pillar 4] Evaluating unseen attack family: "
    f"{CONFIG['zero_day_class']}..."
)
print("=" * 115)

if "df_zero_day_isolated" not in globals():
    raise RuntimeError(
        "Zero-day holdout is unavailable. Run Block 5 first."
    )

if len(df_zero_day_isolated) == 0:
    raise RuntimeError(
        "Zero-day holdout contains no rows."
    )

X_zday = (
    df_zero_day_isolated[expected_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)

y_zday = np.full(
    len(df_zero_day_isolated),
    ZERO_DAY_ENCODED_LABEL,
    dtype=np.int32,
)

zday_metrics = evaluate_selective_operational_environment(
    model=lgb_model,
    idx_calibrators=calibrators,
    reference_signatures=train_signatures,
    tp=TAU_P,
    ts=TAU_S,
    X_matrix=X_zday,
    y_matrix=y_zday,
    environment_tag=(
        f"Zero-Day ({CONFIG['zero_day_class']} Holdout)"
    ),
    benign_prob_tau=STRICT_BENIGN_PROB_TAU,
    benign_sim_tau=STRICT_BENIGN_SIM_TAU,
    probability_fn=get_calibrated_probabilities,
    return_decisions=False,
)

comprehensive_metrics_log.append(zday_metrics)

block_13_df = pd.DataFrame([zday_metrics])

display_columns = [
    "Target_Environment",
    "Attack_Not_Missed_Rate_ANMR",
    "False_Block_Rate_FBR",
    "Escalation_Rate",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
    "RAM_Footprint_Overhead_MB",
]

print("\n" + "#" * 115)
print(" PILLAR 4: ZERO-DAY SELECTIVE-ATTRIBUTION RESULTS ")
print("#" * 115)
print(
    block_13_df[
        [c for c in display_columns if c in block_13_df.columns]
    ].to_string(index=False)
)
print("#" * 115)


In [ ]:

# DIAGNOSTIC SUB-CELL: ZERO-DAY SELECTIVE ROUTING PROFILE
import numpy as np
import pandas as pd

print("=" * 90)
print(" ZERO-DAY SELECTIVE THREE-TIER ROUTING PROFILE ")
print("=" * 90)

if "df_zero_day_isolated" in globals() and len(df_zero_day_isolated) > 0:
    X_diag = (
        df_zero_day_isolated[expected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    y_diag = np.full(
        len(df_zero_day_isolated),
        ZERO_DAY_ENCODED_LABEL,
        dtype=np.int32,
    )

    result = evaluate_selective_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_diag,
        y_matrix=y_diag,
        environment_tag=f"Zero-Day: {CONFIG['zero_day_class']}",
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        probability_fn=get_calibrated_probabilities,
        return_decisions=True,
    )

    log = result.pop("_Decision_Log")

    print(log["Tier"].value_counts().sort_index().to_string())

    print("\nClassifier mapping of the unseen class:")
    print(log["Predicted_Class"].value_counts().to_string())

    print(
        f"\nAttribution evaluated fraction: "
        f"{result['Attribution_Evaluated_Fraction']:.4f}"
    )
    print(
        f"Escalation rate: "
        f"{result['Escalation_Rate']:.4f}"
    )
    print(
        f"ANMR: "
        f"{result['Attack_Not_Missed_Rate_ANMR']:.6f}"
    )
else:
    print("[Diagnostic Error] Zero-day holdout matrix is unavailable.")


In [ ]:
# BLOCK 14: PILLAR 5 — CONSTRAINED BLACK-BOX ADVERSARIAL EVASION ROBUSTNESS
import numpy as np
import pandas as pd
import sys

print("=" * 115)
print("[Pillar 5] Launching constrained black-box adversarial feature-evasion robustness evaluation...")
print("=" * 115)


def build_adversarial_feature_constraints(
    train_frame,
    feature_names,
):
    """Build source-only feature constraints for feasible tabular perturbations."""
    numeric = (
        train_frame[feature_names]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    lower = numeric.quantile(0.001).fillna(0.0).to_numpy(dtype=float)
    upper = numeric.quantile(0.999).fillna(0.0).to_numpy(dtype=float)
    median = numeric.median().fillna(0.0).to_numpy(dtype=float)

    q1 = numeric.quantile(0.25).fillna(0.0).to_numpy(dtype=float)
    q3 = numeric.quantile(0.75).fillna(0.0).to_numpy(dtype=float)
    robust_scale = np.maximum(
        q3 - q1,
        1e-6,
    )

    benign_numeric = (
        train_frame.loc[
            train_frame["label"].astype(str) == "Benign",
            feature_names,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    benign_median = (
        benign_numeric.median()
        .fillna(pd.Series(median, index=feature_names))
        .to_numpy(dtype=float)
    )

    # Immutable/protocol-defining features should not be modified by the attack.
    immutable_patterns = (
        "protocol",
        "flag",
    )

    immutable_mask = np.array([
        any(pattern in str(name).lower() for pattern in immutable_patterns)
        for name in feature_names
    ], dtype=bool)

    # Count-like fields remain discrete after perturbation.
    discrete_patterns = (
        "count",
        "packets",
        "bytes",
        "header",
        "win",
        "bulk",
        "subflow",
    )

    discrete_mask = np.array([
        any(pattern in str(name).lower() for pattern in discrete_patterns)
        for name in feature_names
    ], dtype=bool)

    # Features whose observed lower bound is non-negative remain non-negative.
    nonnegative_mask = lower >= 0.0

    return {
        "lower": lower,
        "upper": upper,
        "median": median,
        "benign_median": benign_median,
        "robust_scale": robust_scale,
        "immutable_mask": immutable_mask,
        "discrete_mask": discrete_mask,
        "nonnegative_mask": nonnegative_mask,
    }


def constrained_black_box_evasion(
    model,
    idx_calibrators,
    X_input,
    y_input,
    epsilon,
    constraints,
    probability_fn=None,
    random_state=42,
    random_candidates=4,
):
    """Vectorized black-box evasion attack against malicious rows only.

    Each candidate moves mutable features toward the benign source median with
    a maximum epsilon-scaled robust perturbation. A small number of randomized
    masks are evaluated in batches, and the candidate with the highest benign
    probability is retained for each malicious flow.

    Benign rows are never perturbed, allowing FBR to remain interpretable.
    """
    probability_fn = probability_fn or get_calibrated_probabilities

    X = np.asarray(X_input, dtype=np.float32)
    y = np.asarray(y_input)

    if float(epsilon) <= 0.0:
        return X.copy()

    attack_mask = y != BENIGN_IDX
    attack_idx = np.where(attack_mask)[0]

    if len(attack_idx) == 0:
        return X.copy()

    X_adv = X.copy()
    X_attack = X[attack_idx].astype(np.float64)

    lower = constraints["lower"]
    upper = constraints["upper"]
    benign_median = constraints["benign_median"]
    robust_scale = constraints["robust_scale"]
    immutable_mask = constraints["immutable_mask"]
    discrete_mask = constraints["discrete_mask"]
    nonnegative_mask = constraints["nonnegative_mask"]

    mutable_mask = ~immutable_mask

    direction = np.sign(
        benign_median[None, :]
        - X_attack
    )

    max_step = (
        float(epsilon)
        * robust_scale[None, :]
    )

    rng = np.random.default_rng(random_state)

    candidate_bank = []

    # Deterministic strongest move toward the benign source center.
    deterministic = X_attack.copy()
    deterministic[:, mutable_mask] += (
        direction[:, mutable_mask]
        * max_step[:, mutable_mask]
    )
    candidate_bank.append(deterministic)

    # Random sparse variants simulate an attacker modifying only subsets of
    # feasible flow statistics.
    for _ in range(max(0, int(random_candidates) - 1)):
        sparse_mask = (
            rng.random(X_attack.shape)
            <= 0.50
        )
        sparse_mask[:, immutable_mask] = False

        candidate = X_attack.copy()
        candidate += (
            direction
            * max_step
            * sparse_mask
        )
        candidate_bank.append(candidate)

    best_candidate = X_attack.copy()
    base_probs = probability_fn(
        model,
        idx_calibrators,
        X_attack.astype(np.float32),
    )
    best_benign_probability = base_probs[:, BENIGN_IDX].copy()

    for candidate in candidate_bank:
        candidate = np.clip(
            candidate,
            lower[None, :],
            upper[None, :],
        )

        candidate[:, nonnegative_mask] = np.maximum(
            candidate[:, nonnegative_mask],
            0.0,
        )

        candidate[:, discrete_mask] = np.round(
            candidate[:, discrete_mask]
        )

        candidate[:, immutable_mask] = X_attack[:, immutable_mask]

        candidate32 = candidate.astype(np.float32)

        probs = probability_fn(
            model,
            idx_calibrators,
            candidate32,
        )
        benign_probability = probs[:, BENIGN_IDX]

        improve = (
            benign_probability
            > best_benign_probability
        )

        if np.any(improve):
            best_candidate[improve] = candidate[improve]
            best_benign_probability[improve] = benign_probability[improve]

    X_adv[attack_idx] = best_candidate.astype(np.float32)
    return X_adv


# Source-only constraints: no held-out labels are used to learn perturbation rules.
ADVERSARIAL_CONSTRAINTS = build_adversarial_feature_constraints(
    train_df,
    expected_features,
)

# Use the Hard-Test partition as the adversarial base because it already combines
# host and temporal shift. This gives a deliberately difficult evasion scenario.
_adversarial_base_df = hard_test_df.copy()

if len(_adversarial_base_df) > int(CONFIG.get("adversarial_eval_max_rows", 20000)):
    _adversarial_base_df = _adversarial_base_df.sample(
        n=int(CONFIG.get("adversarial_eval_max_rows", 20000)),
        random_state=RANDOM_STATE,
        stratify=None,
    )

X_adversarial_base = (
    _adversarial_base_df[expected_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)

y_adversarial_base = encode_active_labels(
    _adversarial_base_df["label"],
    zero_day_value=None,
)

ADVERSARIAL_EPSILONS = list(
    CONFIG.get(
        "adversarial_epsilons",
        [0.00, 0.01, 0.025, 0.05, 0.10],
    )
)

adversarial_records = []
ADVERSARIAL_CAPTURED_MATRICES = {}

for epsilon in ADVERSARIAL_EPSILONS:
    X_adv = constrained_black_box_evasion(
        model=lgb_model,
        idx_calibrators=calibrators,
        X_input=X_adversarial_base,
        y_input=y_adversarial_base,
        epsilon=epsilon,
        constraints=ADVERSARIAL_CONSTRAINTS,
        probability_fn=get_calibrated_probabilities,
        random_state=RANDOM_STATE + int(round(float(epsilon) * 10000)),
        random_candidates=CONFIG.get(
            "adversarial_random_candidates",
            4,
        ),
    )

    ADVERSARIAL_CAPTURED_MATRICES[float(epsilon)] = (
        X_adv.copy(),
        y_adversarial_base.copy(),
    )

    # Classifier-only evasion diagnostic.
    adv_probs = get_calibrated_probabilities(
        lgb_model,
        calibrators,
        X_adv,
    )
    adv_preds = np.argmax(
        adv_probs,
        axis=1,
    )

    malicious_mask = (
        y_adversarial_base
        != BENIGN_IDX
    )
    classifier_evasion_rate = float(
        np.mean(
            adv_preds[malicious_mask]
            == BENIGN_IDX
        )
    ) if np.any(malicious_mask) else 0.0

    metrics = evaluate_selective_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_adv,
        y_matrix=y_adversarial_base,
        environment_tag=(
            f"Adversarial Evasion | epsilon={epsilon:.3f}"
        ),
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        probability_fn=get_calibrated_probabilities,
        return_decisions=False,
    )

    metrics["Adversarial_Epsilon"] = float(epsilon)
    metrics["Classifier_Evasion_Rate"] = classifier_evasion_rate

    adversarial_records.append(metrics)
    comprehensive_metrics_log.append(metrics)

adversarial_results_df = pd.DataFrame(
    adversarial_records
)

print("\n" + "#" * 115)
print(" PILLAR 5: CONSTRAINED BLACK-BOX ADVERSARIAL EVASION ROBUSTNESS ")
print("#" * 115)

display_columns = [
    "Adversarial_Epsilon",
    "Global_Accuracy",
    "Macro_F1_Throughput",
    "Classifier_Evasion_Rate",
    "Attack_Not_Missed_Rate_ANMR",
    "False_Block_Rate_FBR",
    "Escalation_Rate",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
    "RAM_Footprint_Overhead_MB",
]

print(
    adversarial_results_df[
        [c for c in display_columns if c in adversarial_results_df.columns]
    ].to_string(
        index=False,
        formatters={
            "Adversarial_Epsilon": "{:.3f}".format,
            "Global_Accuracy": "{:.4f}".format,
            "Macro_F1_Throughput": "{:.4f}".format,
            "Classifier_Evasion_Rate": "{:.4f}".format,
            "Attack_Not_Missed_Rate_ANMR": "{:.4f}".format,
            "False_Block_Rate_FBR": "{:.4f}".format,
            "Escalation_Rate": "{:.4f}".format,
            "Attribution_Evaluated_Fraction": "{:.4f}".format,
            "Probability_Latency_ms_per_flow": "{:.4f}".format,
            "Attribution_Latency_ms_per_flow": "{:.4f}".format,
            "End_to_End_Latency_ms_per_flow": "{:.4f}".format,
            "End_to_End_Throughput_flows_per_sec": "{:.1f}".format,
            "RAM_Footprint_Overhead_MB": "{:.2f}".format,
        },
    )
)

print("#" * 115)
sys.stdout.flush()


In [ ]:

# BLOCK 15: PILLAR 5 DIAGNOSTIC — ADVERSARIAL DEGRADATION AND SELECTIVE-POLICY RETENTION
import pandas as pd
import numpy as np

if "adversarial_results_df" not in globals() or adversarial_results_df.empty:
    raise RuntimeError(
        "Run Block 14 adversarial robustness evaluation first."
    )

diagnostic_cols = [
    "Adversarial_Epsilon",
    "Classifier_Evasion_Rate",
    "Global_Accuracy",
    "Macro_F1_Throughput",
    "Attack_Not_Missed_Rate_ANMR",
    "False_Block_Rate_FBR",
    "Escalation_Rate",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
]

adversarial_diagnostic_df = adversarial_results_df[
    [c for c in diagnostic_cols if c in adversarial_results_df.columns]
].copy()

clean_row = adversarial_diagnostic_df[
    adversarial_diagnostic_df["Adversarial_Epsilon"] == 0.0
]

if not clean_row.empty:
    clean_anmr = float(
        clean_row.iloc[0]["Attack_Not_Missed_Rate_ANMR"]
    )
    adversarial_diagnostic_df["ANMR_Drop_From_Clean"] = (
        clean_anmr
        - adversarial_diagnostic_df[
            "Attack_Not_Missed_Rate_ANMR"
        ]
    )

print("\n" + "=" * 115)
print("ADVERSARIAL SELECTIVE-POLICY RETENTION DIAGNOSTIC")
print("=" * 115)
print(adversarial_diagnostic_df.to_string(index=False))
print("=" * 115)


In [ ]:

# FINAL VALIDATION-DERIVED POLICY LOCK
print("=" * 115)
print("[Final Selective-Attribution Policy Lock]")
print(f" -> TAU_P                       : {TAU_P:.4f}")
print(f" -> TAU_S                       : {TAU_S:.4f}")
print(f" -> BENIGN_PROB_TAU             : {STRICT_BENIGN_PROB_TAU:.4f}")
print(f" -> BENIGN_SIM_TAU              : {STRICT_BENIGN_SIM_TAU:.4f}")
print(f" -> OVERRIDE_PROB_TAU           : {STRICT_OVERRIDE_PROB_TAU:.4f}")
print(f" -> OVERRIDE_SIM_FLOOR          : {STRICT_OVERRIDE_SIM_FLOOR:.4f}")
print(
    f" -> BENIGN_DIRECT_PROB_TAU      : "
    f"{SELECTIVE_BENIGN_DIRECT_PROB_TAU:.4f}"
)
print(
    f" -> BENIGN_DIRECT_MARGIN_TAU    : "
    f"{SELECTIVE_BENIGN_DIRECT_MARGIN_TAU:.4f}"
)
print(
    f" -> ATTACK_DIRECT_PROB_TAU      : "
    f"{SELECTIVE_ATTACK_DIRECT_PROB_TAU:.4f}"
)
print(
    f" -> ATTACK_DIRECT_MARGIN_TAU    : "
    f"{SELECTIVE_ATTACK_DIRECT_MARGIN_TAU:.4f}"
)
print(" -> Tier 1                       : ALLOW")
print(" -> Tier 2                       : AUTONOMOUS BLOCK")
print(" -> Tier 3                       : ADMINISTRATIVE ESCALATION")
print(" -> Explanation mode             : SELECTIVE SHAP/COSINE VERIFICATION")
print(
    " -> Policy scope                : Host drift, temporal drift, hard drift, "
    "zero-day, adversarial evasion, and multi-seed robustness"
)
print("=" * 115)


In [ ]:

# BLOCK 15B: SELECTIVE-ATTRIBUTION LATENCY AND THROUGHPUT AUDIT
import pandas as pd
import numpy as np

if not comprehensive_metrics_log:
    raise RuntimeError(
        "No selective-policy evaluation metrics are available yet."
    )

latency_audit_df = pd.DataFrame(
    comprehensive_metrics_log
)

dedup_subset = ["Target_Environment"]
if "Adversarial_Epsilon" in latency_audit_df.columns:
    dedup_subset.append("Adversarial_Epsilon")

latency_audit_df = (
    latency_audit_df
    .drop_duplicates(
        subset=dedup_subset,
        keep="last",
    )
    .reset_index(drop=True)
)

latency_columns = [
    "Target_Environment",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
    "RAM_Footprint_Overhead_MB",
]

latency_columns = [
    c for c in latency_columns
    if c in latency_audit_df.columns
]

print("\n" + "=" * 115)
print("SELECTIVE-ATTRIBUTION COMPUTATIONAL AUDIT")
print("=" * 115)
print(
    latency_audit_df[
        latency_columns
    ].to_string(index=False)
)
print("=" * 115)
print(
    "\nAttribution latency is amortized over all input flows; "
    "Attribution_Evaluated_Fraction reports the fraction that actually invoke SHAP."
)


In [ ]:

# BLOCK 15C: THROUGHPUT / SAFETY ABLATION
# 100%-SHAP baseline vs validation-optimized selective attribution.
#
# Run after:
#   - Block 10B (selective thresholds)
#   - Block 11 and 11B (both evaluators)
#   - natural drift / zero-day / adversarial blocks
#
# This block reruns BOTH policies on the same matrices so the timing
# comparison is not taken from different notebook executions.

import numpy as np
import pandas as pd

print("=" * 115)
print("SELECTIVE ATTRIBUTION THROUGHPUT / SAFETY ABLATION")
print("=" * 115)

ablation_environments = {
    "Host-Test": (
        host_test_df[expected_features].to_numpy(dtype=np.float32),
        encode_active_labels(host_test_df["label"], zero_day_value=None),
    ),
    "Time-Test": (
        time_test_df[expected_features].to_numpy(dtype=np.float32),
        encode_active_labels(time_test_df["label"], zero_day_value=None),
    ),
    "Hard-Test": (
        hard_test_df[expected_features].to_numpy(dtype=np.float32),
        encode_active_labels(hard_test_df["label"], zero_day_value=None),
    ),
}

if "df_zero_day_isolated" in globals() and len(df_zero_day_isolated) > 0:
    ablation_environments["Zero-Day"] = (
        df_zero_day_isolated[expected_features].to_numpy(dtype=np.float32),
        np.full(
            len(df_zero_day_isolated),
            ZERO_DAY_ENCODED_LABEL,
            dtype=np.int32,
        ),
    )

if (
    "ADVERSARIAL_CAPTURED_MATRICES" in globals()
    and ADVERSARIAL_CAPTURED_MATRICES
):
    strongest_epsilon = max(ADVERSARIAL_CAPTURED_MATRICES.keys())
    ablation_environments[
        f"Adversarial epsilon={strongest_epsilon:.3f}"
    ] = ADVERSARIAL_CAPTURED_MATRICES[strongest_epsilon]

ablation_records = []

for environment_name, (X_eval_ablation, y_eval_ablation) in (
    ablation_environments.items()
):
    print("\n" + "-" * 115)
    print(f"[Ablation Environment] {environment_name}")
    print("-" * 115)

    # Universal explanation baseline.
    baseline = evaluate_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_eval_ablation,
        y_matrix=y_eval_ablation,
        environment_tag=f"{environment_name} | 100%-SHAP",
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        return_decisions=False,
    )

    baseline_row = {
        "Environment": environment_name,
        "Policy": "100%-SHAP",
        "ANMR": baseline["Attack_Not_Missed_Rate_ANMR"],
        "FBR": baseline["False_Block_Rate_FBR"],
        "Escalation_Rate": baseline["Escalation_Rate"],
        "Probability_Latency_ms_per_flow":
            baseline["Probability_Latency_ms_per_flow"],
        "Attribution_Latency_ms_per_flow":
            baseline["Attribution_Latency_ms_per_flow"],
        "End_to_End_Latency_ms_per_flow":
            baseline["End_to_End_Latency_ms_per_flow"],
        "End_to_End_Throughput_flows_per_sec":
            baseline["End_to_End_Throughput_flows_per_sec"],
        "Attribution_Evaluated_Fraction": 1.0,
    }
    ablation_records.append(baseline_row)

    # Validation-optimized selective explanation policy.
    selective = evaluate_selective_operational_environment(
        model=lgb_model,
        idx_calibrators=calibrators,
        reference_signatures=train_signatures,
        tp=TAU_P,
        ts=TAU_S,
        X_matrix=X_eval_ablation,
        y_matrix=y_eval_ablation,
        environment_tag=f"{environment_name} | Selective",
        benign_prob_tau=STRICT_BENIGN_PROB_TAU,
        benign_sim_tau=STRICT_BENIGN_SIM_TAU,
        return_decisions=False,
    )

    selective_row = {
        "Environment": environment_name,
        "Policy": "Selective",
        "ANMR": selective["Attack_Not_Missed_Rate_ANMR"],
        "FBR": selective["False_Block_Rate_FBR"],
        "Escalation_Rate": selective["Escalation_Rate"],
        "Probability_Latency_ms_per_flow":
            selective["Probability_Latency_ms_per_flow"],
        "Attribution_Latency_ms_per_flow":
            selective["Attribution_Latency_ms_per_flow"],
        "End_to_End_Latency_ms_per_flow":
            selective["End_to_End_Latency_ms_per_flow"],
        "End_to_End_Throughput_flows_per_sec":
            selective["End_to_End_Throughput_flows_per_sec"],
        "Attribution_Evaluated_Fraction":
            selective["Attribution_Evaluated_Fraction"],
    }
    ablation_records.append(selective_row)

SELECTIVE_ATTRIBUTION_ABLATION_DF = pd.DataFrame(ablation_records)

# Compute environment-wise gain relative to the freshly rerun baseline.
baseline_lookup = (
    SELECTIVE_ATTRIBUTION_ABLATION_DF[
        SELECTIVE_ATTRIBUTION_ABLATION_DF["Policy"] == "100%-SHAP"
    ]
    .set_index("Environment")
)

def _baseline_metric(environment, column):
    return float(baseline_lookup.loc[environment, column])

SELECTIVE_ATTRIBUTION_ABLATION_DF["Throughput_Gain_x"] = (
    SELECTIVE_ATTRIBUTION_ABLATION_DF.apply(
        lambda row: (
            row["End_to_End_Throughput_flows_per_sec"]
            / max(
                _baseline_metric(
                    row["Environment"],
                    "End_to_End_Throughput_flows_per_sec",
                ),
                1e-12,
            )
        ),
        axis=1,
    )
)

SELECTIVE_ATTRIBUTION_ABLATION_DF["Latency_Reduction_pct"] = (
    SELECTIVE_ATTRIBUTION_ABLATION_DF.apply(
        lambda row: (
            100.0
            * (
                1.0
                - row["End_to_End_Latency_ms_per_flow"]
                / max(
                    _baseline_metric(
                        row["Environment"],
                        "End_to_End_Latency_ms_per_flow",
                    ),
                    1e-12,
                )
            )
        ),
        axis=1,
    )
)

display_cols = [
    "Environment",
    "Policy",
    "ANMR",
    "FBR",
    "Escalation_Rate",
    "Attribution_Evaluated_Fraction",
    "Probability_Latency_ms_per_flow",
    "Attribution_Latency_ms_per_flow",
    "End_to_End_Latency_ms_per_flow",
    "End_to_End_Throughput_flows_per_sec",
    "Throughput_Gain_x",
    "Latency_Reduction_pct",
]

print(
    SELECTIVE_ATTRIBUTION_ABLATION_DF[
        display_cols
    ].to_string(
        index=False,
        formatters={
            "ANMR": "{:.6f}".format,
            "FBR": "{:.6f}".format,
            "Escalation_Rate": "{:.6f}".format,
            "Attribution_Evaluated_Fraction": "{:.4f}".format,
            "Probability_Latency_ms_per_flow": "{:.4f}".format,
            "Attribution_Latency_ms_per_flow": "{:.4f}".format,
            "End_to_End_Latency_ms_per_flow": "{:.4f}".format,
            "End_to_End_Throughput_flows_per_sec": "{:.2f}".format,
            "Throughput_Gain_x": "{:.2f}".format,
            "Latency_Reduction_pct": "{:.2f}".format,
        },
    )
)

print("\nInterpretation:")
print(
    " - The 100%-SHAP row is the original policy and must remain the safety baseline."
)
print(
    " - The Selective row computes SHAP only for validation-defined unresolved flows."
)
print(
    " - A useful optimization must materially reduce Attribution_Evaluated_Fraction "
    "and latency while preserving ANMR/FBR."
)
print(
    " - Do not adopt the selective policy as the manuscript's final policy until "
    "this table confirms that the throughput gain does not create an unacceptable "
    "safety loss, especially on Zero-Day and Adversarial environments."
)
print("=" * 115)


In [ ]:

# BLOCK 16: SIX-PILLAR SELECTIVE-ATTRIBUTION MASTER RESULTS SUMMARY
import pandas as pd
import numpy as np

print("=" * 115)
print("FINAL SIX-PILLAR SELECTIVE-ATTRIBUTION EXPERIMENTAL BLUEPRINT")
print("=" * 115)

pillar_map = pd.DataFrame([
    {
        "Pillar": 1,
        "Experiment": "Host-based natural concept drift",
        "Purpose": "Topology/host generalization",
        "Policy": "Frozen selective-attribution policy",
    },
    {
        "Pillar": 2,
        "Experiment": "Temporal natural concept drift",
        "Purpose": "Time-based generalization",
        "Policy": "Frozen selective-attribution policy",
    },
    {
        "Pillar": 3,
        "Experiment": "Hard concept drift",
        "Purpose": "Combined host-temporal shift",
        "Policy": "Frozen selective-attribution policy",
    },
    {
        "Pillar": 4,
        "Experiment": "Zero-day attack holdout",
        "Purpose": "Unseen attack-family behavior",
        "Policy": "Frozen selective-attribution policy",
    },
    {
        "Pillar": 5,
        "Experiment": "Adversarial evasion robustness",
        "Purpose": "Constrained malicious feature manipulation",
        "Policy": "Frozen selective-attribution policy",
    },
    {
        "Pillar": 6,
        "Experiment": "Multi-seed robustness",
        "Purpose": "Training-randomness sensitivity",
        "Policy": "Frozen selective thresholds; seed-specific model/calibration/signatures",
    },
])

print(pillar_map.to_string(index=False))
print("=" * 115)

if comprehensive_metrics_log:
    master_results_df = pd.DataFrame(
        comprehensive_metrics_log
    )

    dedup_subset = ["Target_Environment"]
    if "Adversarial_Epsilon" in master_results_df.columns:
        dedup_subset.append("Adversarial_Epsilon")

    master_results_df = (
        master_results_df
        .drop_duplicates(
            subset=dedup_subset,
            keep="last",
        )
        .reset_index(drop=True)
    )

    core_cols = [
        "Target_Environment",
        "Adversarial_Epsilon",
        "Global_Accuracy",
        "Macro_F1_Throughput",
        "Attack_Not_Missed_Rate_ANMR",
        "False_Block_Rate_FBR",
        "Escalation_Rate",
        "Attribution_Evaluated_Fraction",
        "Probability_Latency_ms_per_flow",
        "Attribution_Latency_ms_per_flow",
        "End_to_End_Latency_ms_per_flow",
        "End_to_End_Throughput_flows_per_sec",
        "RAM_Footprint_Overhead_MB",
    ]

    core_cols = [
        c for c in core_cols
        if c in master_results_df.columns
    ]

    print("\n[Final Selective-Policy Single-Run Results]")
    print(
        master_results_df[
            core_cols
        ].to_string(index=False)
    )
else:
    print(
        "\nNo selective-policy evaluation metrics are currently available."
    )


In [ ]:
# BLOCK 16B: ATTRIBUTION-SIMILARITY METRIC ABLATION
# Compares cosine, Pearson, Spearman, and Euclidean similarity.

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import roc_auc_score

print("=" * 115)
print("ATTRIBUTION-SIMILARITY METRIC ABLATION")
print("=" * 115)


# --------------------------------------------------------------------------------------
# Similarity functions
# --------------------------------------------------------------------------------------

def safe_cosine_similarity(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=float)
    b = np.asarray(vector_b, dtype=float)

    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    if denominator <= 1e-12:
        return 0.0

    return float(
        np.dot(a, b) / denominator
    )


def safe_pearson_similarity(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=float)
    b = np.asarray(vector_b, dtype=float)

    if (
        len(a) < 2
        or np.std(a) <= 1e-12
        or np.std(b) <= 1e-12
    ):
        return 0.0

    value = pearsonr(a, b).statistic

    if not np.isfinite(value):
        return 0.0

    return float(value)


def safe_spearman_similarity(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=float)
    b = np.asarray(vector_b, dtype=float)

    if (
        len(a) < 2
        or np.std(a) <= 1e-12
        or np.std(b) <= 1e-12
    ):
        return 0.0

    value = spearmanr(
        a,
        b
    ).correlation

    if not np.isfinite(value):
        return 0.0

    return float(value)


def euclidean_to_similarity(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=float)
    b = np.asarray(vector_b, dtype=float)

    distance = np.linalg.norm(
        a - b
    )

    return float(
        1.0 / (1.0 + distance)
    )


SIMILARITY_FUNCTIONS = {
    "Cosine": safe_cosine_similarity,
    "Pearson": safe_pearson_similarity,
    "Spearman": safe_spearman_similarity,
    "Euclidean": euclidean_to_similarity,
}


# --------------------------------------------------------------------------------------
# Evaluation environments
# --------------------------------------------------------------------------------------

ablation_sources = {
    "Host-Test": host_test_df,
    "Time-Test": time_test_df,
    "Hard-Test": hard_test_df,
}

# Add strongest adversarial condition when available.
if (
    "ADVERSARIAL_CAPTURED_MATRICES" in globals()
    and ADVERSARIAL_CAPTURED_MATRICES
):
    strongest_epsilon = max(
        ADVERSARIAL_CAPTURED_MATRICES.keys()
    )

    X_adv_ablation, y_adv_ablation = (
        ADVERSARIAL_CAPTURED_MATRICES[
            strongest_epsilon
        ]
    )
else:
    strongest_epsilon = None
    X_adv_ablation = None
    y_adv_ablation = None


# --------------------------------------------------------------------------------------
# Sample-level metric evaluation
# --------------------------------------------------------------------------------------

metric_records = []
sample_records = []

for environment_name, frame in ablation_sources.items():

    X_eval = (
        frame[expected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    y_eval = encode_active_labels(
        frame["label"],
        zero_day_value=None,
    )

    probabilities = get_calibrated_probabilities(
        lgb_model,
        calibrators,
        X_eval,
    )

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    predicted_names = np.array(
        [
            resolve_active_class_name(index)
            for index in predictions
        ],
        dtype=object,
    )

    # Existing utility is reused to obtain normalized local attributions.
    _, normalized_attributions = (
        compute_predicted_signature_similarity(
            lgb_model,
            train_signatures,
            X_eval,
            predictions,
            predicted_names,
            batch_size=CONFIG.get(
                "attribution_batch_size",
                8192,
            ),
        )
    )

    for row_index in range(len(X_eval)):

        predicted_name = predicted_names[
            row_index
        ]

        true_name = resolve_active_class_name(
            y_eval[row_index]
        )

        predicted_reference = train_signatures.get(
            predicted_name
        )

        true_reference = train_signatures.get(
            true_name
        )

        if (
            predicted_reference is None
            or true_reference is None
        ):
            continue

        attribution_vector = (
            normalized_attributions[
                row_index
            ]
        )

        prediction_correct = int(
            predictions[row_index]
            == y_eval[row_index]
        )

        for metric_name, metric_function in (
            SIMILARITY_FUNCTIONS.items()
        ):

            predicted_similarity = metric_function(
                attribution_vector,
                predicted_reference,
            )

            true_similarity = metric_function(
                attribution_vector,
                true_reference,
            )

            sample_records.append({
                "Environment": environment_name,
                "Metric": metric_name,
                "True_Class": true_name,
                "Predicted_Class": predicted_name,
                "Prediction_Correct": prediction_correct,
                "Predicted_Signature_Similarity":
                    predicted_similarity,
                "True_Signature_Similarity":
                    true_similarity,
            })


# --------------------------------------------------------------------------------------
# Strongest adversarial condition
# --------------------------------------------------------------------------------------

if X_adv_ablation is not None:

    probabilities = get_calibrated_probabilities(
        lgb_model,
        calibrators,
        X_adv_ablation,
    )

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    predicted_names = np.array(
        [
            resolve_active_class_name(index)
            for index in predictions
        ],
        dtype=object,
    )

    _, normalized_attributions = (
        compute_predicted_signature_similarity(
            lgb_model,
            train_signatures,
            X_adv_ablation,
            predictions,
            predicted_names,
            batch_size=CONFIG.get(
                "attribution_batch_size",
                8192,
            ),
        )
    )

    environment_name = (
        f"Adversarial ε={strongest_epsilon:.3f}"
    )

    for row_index in range(
        len(X_adv_ablation)
    ):

        predicted_name = predicted_names[
            row_index
        ]

        true_name = resolve_active_class_name(
            y_adv_ablation[row_index]
        )

        predicted_reference = train_signatures.get(
            predicted_name
        )

        true_reference = train_signatures.get(
            true_name
        )

        if (
            predicted_reference is None
            or true_reference is None
        ):
            continue

        attribution_vector = (
            normalized_attributions[
                row_index
            ]
        )

        prediction_correct = int(
            predictions[row_index]
            == y_adv_ablation[row_index]
        )

        for metric_name, metric_function in (
            SIMILARITY_FUNCTIONS.items()
        ):

            predicted_similarity = metric_function(
                attribution_vector,
                predicted_reference,
            )

            true_similarity = metric_function(
                attribution_vector,
                true_reference,
            )

            sample_records.append({
                "Environment": environment_name,
                "Metric": metric_name,
                "True_Class": true_name,
                "Predicted_Class": predicted_name,
                "Prediction_Correct": prediction_correct,
                "Predicted_Signature_Similarity":
                    predicted_similarity,
                "True_Signature_Similarity":
                    true_similarity,
            })


similarity_ablation_samples_df = pd.DataFrame(
    sample_records
)


# --------------------------------------------------------------------------------------
# Separation analysis
# --------------------------------------------------------------------------------------

for (
    environment_name,
    metric_name
), group in similarity_ablation_samples_df.groupby(
    ["Environment", "Metric"]
):

    correct_scores = group.loc[
        group["Prediction_Correct"] == 1,
        "Predicted_Signature_Similarity",
    ].to_numpy(dtype=float)

    incorrect_scores = group.loc[
        group["Prediction_Correct"] == 0,
        "Predicted_Signature_Similarity",
    ].to_numpy(dtype=float)

    correct_mean = (
        float(np.mean(correct_scores))
        if len(correct_scores)
        else np.nan
    )

    incorrect_mean = (
        float(np.mean(incorrect_scores))
        if len(incorrect_scores)
        else np.nan
    )

    separation_gap = (
        correct_mean - incorrect_mean
        if (
            np.isfinite(correct_mean)
            and np.isfinite(incorrect_mean)
        )
        else np.nan
    )

    # AUC measures how well the similarity separates
    # correct signature matches from incorrect matches.
    labels = group[
        "Prediction_Correct"
    ].to_numpy(dtype=int)

    scores = group[
        "Predicted_Signature_Similarity"
    ].to_numpy(dtype=float)

    if len(np.unique(labels)) == 2:
        separation_auc = roc_auc_score(
            labels,
            scores,
        )
    else:
        separation_auc = np.nan

    metric_records.append({
        "Environment": environment_name,
        "Metric": metric_name,
        "Correct_Mean_Similarity": correct_mean,
        "Incorrect_Mean_Similarity": incorrect_mean,
        "Separation_Gap": separation_gap,
        "Separation_AUC": separation_auc,
        "Evaluated_Rows": int(len(group)),
    })


similarity_metric_ablation_df = pd.DataFrame(
    metric_records
)


# --------------------------------------------------------------------------------------
# Aggregate ranking
# --------------------------------------------------------------------------------------

metric_summary_df = (
    similarity_metric_ablation_df
    .groupby("Metric", as_index=False)
    .agg(
        Mean_Correct_Similarity=(
            "Correct_Mean_Similarity",
            "mean",
        ),
        Mean_Incorrect_Similarity=(
            "Incorrect_Mean_Similarity",
            "mean",
        ),
        Mean_Separation_Gap=(
            "Separation_Gap",
            "mean",
        ),
        Mean_Separation_AUC=(
            "Separation_AUC",
            "mean",
        ),
        Environment_Count=(
            "Environment",
            "nunique",
        ),
    )
)

metric_summary_df = metric_summary_df.sort_values(
    by=[
        "Mean_Separation_AUC",
        "Mean_Separation_Gap",
    ],
    ascending=False,
).reset_index(drop=True)

metric_summary_df[
    "Ablation_Rank"
] = np.arange(
    1,
    len(metric_summary_df) + 1,
)


print("\n[Environment-Level Results]")
print(
    similarity_metric_ablation_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n[Aggregate Metric Ranking]")
print(
    metric_summary_df[
        [
            "Ablation_Rank",
            "Metric",
            "Mean_Correct_Similarity",
            "Mean_Incorrect_Similarity",
            "Mean_Separation_Gap",
            "Mean_Separation_AUC",
            "Environment_Count",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("=" * 115)

In [ ]:
# BLOCK 16C: CLASS-WISE SHAP BEHAVIORAL FEATURE PROFILES
# Reports the top 15 features contributing toward each true class output.

import numpy as np
import pandas as pd
import shap

print("=" * 115)
print("CLASS-WISE SHAP BEHAVIORAL FEATURE PROFILES")
print("=" * 115)

TOP_K_FEATURES = 15
MAX_PROFILE_ROWS = int(
    CONFIG.get("shap_behavior_profile_max_rows", 20000)
)

# -------------------------------------------------------------------------
# Prepare training data
# -------------------------------------------------------------------------

X_profile_df = (
    train_df[expected_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

y_profile = encode_active_labels(
    train_df["label"],
    zero_day_value=None,
)

# Use a reproducible sample to keep SHAP computation manageable.
if len(X_profile_df) > MAX_PROFILE_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)

    # Stratified sampling so every active class is represented.
    sampled_positions = []

    unique_classes, class_counts = np.unique(
        y_profile,
        return_counts=True,
    )

    for class_index, class_count in zip(
        unique_classes,
        class_counts,
    ):
        class_positions = np.where(
            y_profile == class_index
        )[0]

        class_quota = max(
            1,
            int(
                round(
                    MAX_PROFILE_ROWS
                    * class_count
                    / len(y_profile)
                )
            ),
        )

        class_quota = min(
            class_quota,
            len(class_positions),
        )

        sampled_positions.extend(
            rng.choice(
                class_positions,
                size=class_quota,
                replace=False,
            ).tolist()
        )

    sampled_positions = np.asarray(
        sampled_positions,
        dtype=int,
    )

    # Enforce the requested maximum if rounding produced extra rows.
    if len(sampled_positions) > MAX_PROFILE_ROWS:
        sampled_positions = rng.choice(
            sampled_positions,
            size=MAX_PROFILE_ROWS,
            replace=False,
        )

    X_profile_df = X_profile_df.iloc[
        sampled_positions
    ].reset_index(drop=True)

    y_profile = np.asarray(
        y_profile
    )[sampled_positions]

X_profile = X_profile_df.to_numpy(
    dtype=np.float32
)

print(
    f"SHAP profile rows: {len(X_profile):,}"
)

# -------------------------------------------------------------------------
# Compute multiclass SHAP values
# -------------------------------------------------------------------------

explainer = shap.TreeExplainer(
    lgb_model
)

raw_shap_values = explainer.shap_values(
    X_profile
)


def standardize_multiclass_shap(
    raw_values,
    row_count,
    feature_count,
    class_count,
):
    """
    Return SHAP values as:
        rows × classes × features

    Handles common SHAP/LightGBM output formats.
    """

    # Older SHAP versions:
    # list[class] -> rows × features
    if isinstance(raw_values, list):
        values = np.stack(
            [
                np.asarray(class_values)
                for class_values in raw_values
            ],
            axis=1,
        )

    else:
        values = np.asarray(raw_values)

        if values.ndim == 2:
            # Binary/single-output case:
            # rows × features
            values = values[:, None, :]

        elif values.ndim == 3:
            # Common modern format:
            # rows × features × classes
            if (
                values.shape[0] == row_count
                and values.shape[1] == feature_count
                and values.shape[2] == class_count
            ):
                values = np.transpose(
                    values,
                    (0, 2, 1),
                )

            # Already:
            # rows × classes × features
            elif (
                values.shape[0] == row_count
                and values.shape[1] == class_count
                and values.shape[2] == feature_count
            ):
                pass

            else:
                raise ValueError(
                    "Unsupported three-dimensional SHAP shape: "
                    f"{values.shape}"
                )

        else:
            raise ValueError(
                "Unsupported SHAP output shape: "
                f"{values.shape}"
            )

    if values.shape[0] != row_count:
        raise ValueError(
            "SHAP row count does not match input data: "
            f"{values.shape[0]} versus {row_count}."
        )

    if values.shape[2] != feature_count:
        raise ValueError(
            "SHAP feature count does not match expected features: "
            f"{values.shape[2]} versus {feature_count}."
        )

    return values.astype(
        np.float64,
        copy=False,
    )


active_class_count = len(
    np.unique(y_profile)
)

shap_matrix = standardize_multiclass_shap(
    raw_values=raw_shap_values,
    row_count=len(X_profile),
    feature_count=len(expected_features),
    class_count=active_class_count,
)

print(
    "Standardized SHAP shape:",
    shap_matrix.shape,
)

# -------------------------------------------------------------------------
# Build true-class behavioral profiles
# -------------------------------------------------------------------------

behavior_records = []

for class_index in np.unique(y_profile):

    class_index = int(class_index)
    class_name = resolve_active_class_name(
        class_index
    )

    class_mask = (
        y_profile == class_index
    )

    if not np.any(class_mask):
        continue

    if class_index >= shap_matrix.shape[1]:
        print(
            f"[Warning] No SHAP output found for "
            f"class index {class_index} ({class_name})."
        )
        continue

    # Extract contributions toward this class's own model output.
    class_shap = shap_matrix[
        class_mask,
        class_index,
        :,
    ]

    mean_absolute_shap = np.mean(
        np.abs(class_shap),
        axis=0,
    )

    mean_signed_shap = np.mean(
        class_shap,
        axis=0,
    )

    positive_fraction = np.mean(
        class_shap > 0,
        axis=0,
    )

    total_importance = float(
        np.sum(mean_absolute_shap)
    )

    if total_importance > 0:
        contribution_share = (
            mean_absolute_shap
            / total_importance
        )
    else:
        contribution_share = np.zeros_like(
            mean_absolute_shap
        )

    top_indices = np.argsort(
        mean_absolute_shap
    )[::-1][:TOP_K_FEATURES]

    for rank, feature_index in enumerate(
        top_indices,
        start=1,
    ):
        signed_value = float(
            mean_signed_shap[feature_index]
        )

        if signed_value > 0:
            average_direction = (
                "Supports class"
            )
        elif signed_value < 0:
            average_direction = (
                "Opposes class"
            )
        else:
            average_direction = "Neutral"

        behavior_records.append({
            "Class": class_name,
            "Rank": rank,
            "Feature": expected_features[
                feature_index
            ],
            "Mean_Absolute_SHAP": float(
                mean_absolute_shap[
                    feature_index
                ]
            ),
            "Mean_Signed_SHAP": signed_value,
            "Contribution_Share": float(
                contribution_share[
                    feature_index
                ]
            ),
            "Positive_SHAP_Fraction": float(
                positive_fraction[
                    feature_index
                ]
            ),
            "Average_Direction":
                average_direction,
            "Class_Sample_Count": int(
                np.sum(class_mask)
            ),
        })


class_shap_behavior_df = pd.DataFrame(
    behavior_records
)

# -------------------------------------------------------------------------
# Display top 15 features for every class
# -------------------------------------------------------------------------

if class_shap_behavior_df.empty:
    print(
        "No class-wise SHAP behavioral "
        "profiles were generated."
    )

else:
    class_order = (
        class_shap_behavior_df["Class"]
        .drop_duplicates()
        .tolist()
    )

    for class_name in class_order:
        class_table = (
            class_shap_behavior_df.loc[
                class_shap_behavior_df[
                    "Class"
                ] == class_name
            ]
            .sort_values("Rank")
        )

        print("\n" + "-" * 115)
        print(
            f"CLASS: {class_name} | "
            f"TOP {TOP_K_FEATURES} "
            "BEHAVIORAL FEATURES"
        )
        print("-" * 115)

        print(
            class_table[
                [
                    "Rank",
                    "Feature",
                    "Mean_Absolute_SHAP",
                    "Mean_Signed_SHAP",
                    "Contribution_Share",
                    "Positive_SHAP_Fraction",
                    "Average_Direction",
                    "Class_Sample_Count",
                ]
            ].to_string(
                index=False,
                formatters={
                    "Mean_Absolute_SHAP":
                        "{:.6f}".format,
                    "Mean_Signed_SHAP":
                        "{:.6f}".format,
                    "Contribution_Share":
                        "{:.2%}".format,
                    "Positive_SHAP_Fraction":
                        "{:.2%}".format,
                },
            )
        )

print("\n" + "=" * 115)
print(
    "Mean Absolute SHAP ranks overall feature importance. "
    "Mean Signed SHAP indicates whether a feature supports or "
    "opposes the class output on average. Positive SHAP Fraction "
    "shows how consistently the feature supports the class."
)
print("=" * 115)

In [ ]:
# BLOCK 16D: PERSISTENT EXPORT CONSOLIDATOR — SELECTIVE-ATTRIBUTION PIPELINE

import matplotlib.pyplot as plt


print("=" * 115)
print("[System Consolidator] Compiling master operational matrices...")
print("=" * 115)
sys.stdout.flush()

# 1. Instantiate the comprehensive telemetry log from memory space safely
if 'comprehensive_metrics_log' not in globals() and 'comprehensive_metrics_log' not in locals():
    print("[!] Error: No active metrics logs found. Ensure you ran Blocks 12, 13, and 14 first.")
    master_report_df = pd.DataFrame()
else:
    # Convert global collection array to structural DataFrame format
    raw_report_df = pd.DataFrame(comprehensive_metrics_log)

    # --- CRITICAL DE-DUPLICATION FILTER ---
    # Drops residual duplicate entries caused by running validation cells out of order
    master_report_df = raw_report_df.drop_duplicates(subset=['Target_Environment'], keep='last').reset_index(drop=True)

# 2. Complete clean local data file export passes
# Uses safe local root assignments to prevent NameError crashes
local_table_dir = os.path.join(os.getcwd(), 'tables')
local_figure_dir = os.path.join(os.getcwd(), 'figures')
os.makedirs(local_table_dir, exist_ok=True)
os.makedirs(local_figure_dir, exist_ok=True)

local_table_path = os.path.join(local_table_dir, 'selective_attribution_pipeline_metrics.csv')
master_report_df.to_csv(local_table_path, index=False)
print(f"[Workspace Sync] Local metrics table saved safely at: {local_table_path}")

# 3. Synchronize cleanly straight into the unified cloud storage target directory
try:
    drive_destination_path = os.path.join(CONFIG['results_dir'], 'selective_attribution_pipeline_metrics.csv')
    # Make sure target container exists before piping raw files down
    os.makedirs(os.path.dirname(drive_destination_path), exist_ok=True)
    master_report_df.to_csv(drive_destination_path, index=False)
    print("[Google Drive Master Synced] Final tables written cleanly to project directory path at:", drive_destination_path)
except Exception as e:
    print("[Google Drive Sync Alert] Active cloud mount connection trace invisible. Table data preserved in local instance cache runtime workspace.")

# 4. Generate the empirical visualization summary chart for the manuscript
if not master_report_df.empty:
    plt.figure(figsize=(11, 5.5))

    # Render horizontal performance visualization bars cleanly
    bars = plt.barh(
        master_report_df['Target_Environment'],
        master_report_df['Attack_Not_Missed_Rate_ANMR'],
        color='navy', edgecolor='black', height=0.45, alpha=0.9
    )

    # Annotate absolute scores inline for clear reviewer communication
    for bar in bars:
        width = bar.get_width()
        plt.text(
            width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
            va='center', ha='left', fontsize=9, fontweight='bold', color='black'
        )

    plt.xlabel('Attack-Not-Missed Rate (ANMR)', fontsize=11, fontweight='bold', labelpad=10)
    plt.title('Selective-Attribution Containment Across Core Evaluation Spaces', fontsize=12, fontweight='bold', pad=15)
    plt.xlim(0, 1.15)
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.gca().invert_yaxis()  # Keeps chronological pipeline steps reading nicely top-to-bottom
    plt.tight_layout()

    # Safe dual-export path loop protection logic execution block
    local_fig_path = os.path.join(local_figure_dir, 'safety_gate_anmr_generalization.png')
    plt.savefig(local_fig_path, dpi=300)

    try:
        drive_fig_path = os.path.join(CONFIG['results_dir'], 'safety_gate_anmr_generalization.png')
        plt.savefig(drive_fig_path, dpi=300)
        print(f"[Visualization Saved] Summary plot synced to Drive: {drive_fig_path}")
    except Exception:
        print(f"[Visualization Saved] Summary plot written locally: {local_fig_path}")

    plt.show()
    plt.close()

print("\n" + "=" * 105)
print("        ALL EXPERIMENTAL PILLARS RE-ARCHITECTED SUCCESSFULLY FOR MANUSCRIPT RESUBMISSION        ")
print("=" * 105)

# Output your pristine, un-leaked final metric array directly to the notebook terminal log window
if not master_report_df.empty:
    print(master_report_df.to_string(index=False))
print("=" * 105 + "\n")
sys.stdout.flush()


# Export explicit 100%-SHAP versus selective-attribution ablation.
if (
    "SELECTIVE_ATTRIBUTION_ABLATION_DF" in globals()
    and not SELECTIVE_ATTRIBUTION_ABLATION_DF.empty
):
    ablation_local_path = os.path.join(
        local_table_dir,
        "selective_attribution_throughput_safety_ablation.csv",
    )
    SELECTIVE_ATTRIBUTION_ABLATION_DF.to_csv(
        ablation_local_path,
        index=False,
    )
    print(
        "[Workspace Sync] Throughput/safety ablation saved at:",
        ablation_local_path,
    )

    try:
        ablation_drive_path = os.path.join(
            CONFIG["results_dir"],
            "selective_attribution_throughput_safety_ablation.csv",
        )
        SELECTIVE_ATTRIBUTION_ABLATION_DF.to_csv(
            ablation_drive_path,
            index=False,
        )
        print(
            "[Google Drive Master Synced] Ablation table written to:",
            ablation_drive_path,
        )
    except Exception:
        pass


In [ ]:
# BLOCK 17: PILLAR 6 — SELECTIVE-ATTRIBUTION MULTI-SEED ROBUSTNESS ENGINE
# NOTE:
# - Run Block 10B and Block 11B first.
# - Selective thresholds remain frozen from validation.
# - This block varies only the model seed, calibration, and source signatures.

import os
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.calibration import IsotonicRegression

print("=" * 115)
print("[Pillar 6] Initiating selective-attribution multi-seed robustness verification...")
print("=" * 115)

SEEDS = [42, 123, 456]
CLASS_COUNT = len(ACTIVE_CLASS_INDEX)

FINAL_TAU_P = float(TAU_P)
FINAL_TAU_S = float(TAU_S)
FINAL_BENIGN_PROB_TAU = float(STRICT_BENIGN_PROB_TAU)
FINAL_BENIGN_SIM_TAU = float(STRICT_BENIGN_SIM_TAU)

required_selective_thresholds = [
    "SELECTIVE_BENIGN_DIRECT_PROB_TAU",
    "SELECTIVE_BENIGN_DIRECT_MARGIN_TAU",
    "SELECTIVE_ATTACK_DIRECT_PROB_TAU",
    "SELECTIVE_ATTACK_DIRECT_MARGIN_TAU",
]
missing_selective_thresholds = [
    name for name in required_selective_thresholds
    if name not in globals()
]
if missing_selective_thresholds:
    raise RuntimeError(
        "Run Block 10B before Block 17. Missing: "
        + ", ".join(missing_selective_thresholds)
    )

if "evaluate_selective_operational_environment" not in globals():
    raise RuntimeError(
        "Run Block 11B before Block 17."
    )

base_environments = {
    "Host-Test": host_test_df,
    "Time-Test": time_test_df,
    "Hard-Test": hard_test_df,
    "Zero-Day": df_zero_day_isolated.copy(),
}

STRONG_ADVERSARIAL_EPSILON = max(
    CONFIG.get("adversarial_epsilons", [0.10])
)

environment_names = [
    "Host-Test",
    "Time-Test",
    "Hard-Test",
    "Zero-Day",
    f"Adversarial epsilon={STRONG_ADVERSARIAL_EPSILON:.3f}",
]

multi_seed_registry = {
    env: {
        "ANMR": [],
        "FBR": [],
        "Accuracy": [],
        "Macro_F1": [],
        "Escalation": [],
        "Attribution_Fraction": [],
        "Probability_Latency_ms_per_flow": [],
        "Attribution_Latency_ms_per_flow": [],
        "End_to_End_Latency_ms_per_flow": [],
        "Flow_Throughput": [],
        "RAM_Overhead_MB": [],
    }
    for env in environment_names
}


def _row_l2_normalize(matrix):
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(
        matrix,
        norms,
        out=np.zeros_like(matrix),
        where=norms > 1e-9,
    )


def _append_selective_metrics(registry, result):
    metric_map = {
        "ANMR": "Attack_Not_Missed_Rate_ANMR",
        "FBR": "False_Block_Rate_FBR",
        "Accuracy": "Global_Accuracy",
        "Macro_F1": "Macro_F1_Throughput",
        "Escalation": "Escalation_Rate",
        "Attribution_Fraction": "Attribution_Evaluated_Fraction",
        "Probability_Latency_ms_per_flow":
            "Probability_Latency_ms_per_flow",
        "Attribution_Latency_ms_per_flow":
            "Attribution_Latency_ms_per_flow",
        "End_to_End_Latency_ms_per_flow":
            "End_to_End_Latency_ms_per_flow",
        "Flow_Throughput":
            "End_to_End_Throughput_flows_per_sec",
        "RAM_Overhead_MB":
            "RAM_Footprint_Overhead_MB",
    }
    for target_key, source_key in metric_map.items():
        registry[target_key].append(float(result[source_key]))


for current_seed in SEEDS:
    print("\n" + "=" * 115)
    print(f"[Selective Multi-Seed Run] Seed={current_seed}")
    print("=" * 115)

    np.random.seed(current_seed)

    X_train_seed = train_df[expected_features].to_numpy(dtype=np.float32)
    y_train_seed = encode_active_labels(
        train_df["label"],
        zero_day_value=None,
    )

    X_val_seed = val_df[expected_features].to_numpy(dtype=np.float32)
    y_val_seed = encode_active_labels(
        val_df["label"],
        zero_day_value=None,
    )

    train_data_seed = lgb.Dataset(
        X_train_seed,
        label=y_train_seed,
    )
    val_data_seed = lgb.Dataset(
        X_val_seed,
        label=y_val_seed,
        reference=train_data_seed,
    )

    seed_params = {
        "objective": "multiclass",
        "num_class": CLASS_COUNT,
        "metric": "multi_logloss",
        "learning_rate": 0.05,
        "num_leaves": 32,
        "random_state": current_seed,
        "verbose": -1,
        "num_threads": max(1, (os.cpu_count() or 2) - 1),
    }

    seed_model = lgb.train(
        seed_params,
        train_data_seed,
        num_boost_round=100,
        valid_sets=[val_data_seed],
        callbacks=[
            lgb.early_stopping(10, verbose=False)
        ],
    )

    raw_val = seed_model.predict(X_val_seed)

    seed_calibrators = {}
    for class_idx in range(CLASS_COUNT):
        calibrator = IsotonicRegression(out_of_bounds="clip")
        target_binary = (y_val_seed == class_idx).astype(int)

        if len(np.unique(target_binary)) < 2:
            calibrator.fit(
                np.append(raw_val[:, class_idx], [0.0, 1.0]),
                np.append(target_binary, [0, 1]),
            )
        else:
            calibrator.fit(
                raw_val[:, class_idx],
                target_binary,
            )

        seed_calibrators[class_idx] = calibrator

    def seed_probability_fn(model, idx_calibrators, X_mat):
        raw = model.predict(X_mat)
        calibrated = np.zeros_like(raw)

        for cls_idx in range(CLASS_COUNT):
            calibrated[:, cls_idx] = idx_calibrators[
                cls_idx
            ].transform(raw[:, cls_idx])

        sums = calibrated.sum(axis=1, keepdims=True)
        sums[sums == 0] = 1.0
        return calibrated / sums

    # Seed-specific source signatures.
    seed_signatures = {}

    for class_idx in range(CLASS_COUNT):
        class_rows = np.where(y_train_seed == class_idx)[0]
        if len(class_rows) == 0:
            continue

        rng = np.random.default_rng(current_seed + class_idx)
        sampled_rows = rng.choice(
            class_rows,
            size=min(len(class_rows), 40),
            replace=False,
        )

        predicted_indices = np.full(
            len(sampled_rows),
            class_idx,
            dtype=int,
        )

        attributions = fast_predicted_class_attributions(
            seed_model,
            X_train_seed[sampled_rows],
            predicted_indices,
            batch_size=CONFIG.get(
                "attribution_batch_size",
                8192,
            ),
        )

        normalized = _row_l2_normalize(attributions)
        mean_vector = np.mean(normalized, axis=0)

        seed_signatures[
            resolve_active_class_name(class_idx)
        ] = (
            mean_vector
            / (np.linalg.norm(mean_vector) + 1e-9)
        )

    # Host, Time, Hard, Zero-Day.
    for env_name, env_df in base_environments.items():
        X_eval = (
            env_df[expected_features]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .to_numpy(dtype=np.float32)
        )

        y_eval = encode_active_labels(
            env_df["label"],
            zero_day_value=(
                ZERO_DAY_ENCODED_LABEL
                if env_name == "Zero-Day"
                else None
            ),
        )

        result = evaluate_selective_operational_environment(
            model=seed_model,
            idx_calibrators=seed_calibrators,
            reference_signatures=seed_signatures,
            tp=FINAL_TAU_P,
            ts=FINAL_TAU_S,
            X_matrix=X_eval,
            y_matrix=y_eval,
            environment_tag=(
                f"{env_name} | selective | seed={current_seed}"
            ),
            benign_prob_tau=FINAL_BENIGN_PROB_TAU,
            benign_sim_tau=FINAL_BENIGN_SIM_TAU,
            probability_fn=seed_probability_fn,
            return_decisions=False,
        )

        _append_selective_metrics(
            multi_seed_registry[env_name],
            result,
        )

    # Strongest adversarial condition against Hard-Test.
    X_hard_seed = (
        hard_test_df[expected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    y_hard_seed = encode_active_labels(
        hard_test_df["label"],
        zero_day_value=None,
    )

    seed_constraints = build_adversarial_feature_constraints(
        train_df,
        expected_features,
    )

    X_adv_seed = constrained_black_box_evasion(
        model=seed_model,
        idx_calibrators=seed_calibrators,
        X_input=X_hard_seed,
        y_input=y_hard_seed,
        epsilon=STRONG_ADVERSARIAL_EPSILON,
        constraints=seed_constraints,
        probability_fn=seed_probability_fn,
        random_state=current_seed,
        random_candidates=CONFIG.get(
            "adversarial_random_candidates",
            4,
        ),
    )

    adv_name = (
        f"Adversarial epsilon="
        f"{STRONG_ADVERSARIAL_EPSILON:.3f}"
    )

    adv_result = evaluate_selective_operational_environment(
        model=seed_model,
        idx_calibrators=seed_calibrators,
        reference_signatures=seed_signatures,
        tp=FINAL_TAU_P,
        ts=FINAL_TAU_S,
        X_matrix=X_adv_seed,
        y_matrix=y_hard_seed,
        environment_tag=(
            f"{adv_name} | selective | seed={current_seed}"
        ),
        benign_prob_tau=FINAL_BENIGN_PROB_TAU,
        benign_sim_tau=FINAL_BENIGN_SIM_TAU,
        probability_fn=seed_probability_fn,
        return_decisions=False,
    )

    _append_selective_metrics(
        multi_seed_registry[adv_name],
        adv_result,
    )

    del seed_model, seed_calibrators, seed_signatures
    gc.collect()


def _safe_mean(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    return float(np.mean(arr[finite])) if np.any(finite) else np.nan


def _safe_std(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    return float(np.std(arr[finite])) if np.any(finite) else np.nan


multi_seed_rows = []

for env_name, metrics in multi_seed_registry.items():
    row = {"Environment": env_name}

    for metric_name, values in metrics.items():
        row[f"{metric_name}_Mean"] = _safe_mean(values)
        row[f"{metric_name}_Std"] = _safe_std(values)

    multi_seed_rows.append(row)


multi_seed_results_df = pd.DataFrame(
    multi_seed_rows
)

display_columns = [
    "Environment",
    "ANMR_Mean",
    "ANMR_Std",
    "FBR_Mean",
    "FBR_Std",
    "Accuracy_Mean",
    "Accuracy_Std",
    "Macro_F1_Mean",
    "Macro_F1_Std",
    "Escalation_Mean",
    "Escalation_Std",
    "Attribution_Fraction_Mean",
    "Attribution_Fraction_Std",
    "Probability_Latency_ms_per_flow_Mean",
    "Probability_Latency_ms_per_flow_Std",
    "Attribution_Latency_ms_per_flow_Mean",
    "Attribution_Latency_ms_per_flow_Std",
    "End_to_End_Latency_ms_per_flow_Mean",
    "End_to_End_Latency_ms_per_flow_Std",
    "Flow_Throughput_Mean",
    "Flow_Throughput_Std",
    "RAM_Overhead_MB_Mean",
    "RAM_Overhead_MB_Std",
]

display_columns = [
    c for c in display_columns
    if c in multi_seed_results_df.columns
]

print("\n" + "=" * 115)
print(
    "PILLAR 6: SELECTIVE-ATTRIBUTION MULTI-SEED ROBUSTNESS MATRIX — MEAN ± STD"
)
print("=" * 115)

print(
    multi_seed_results_df[
        display_columns
    ].to_string(index=False)
)

print("=" * 115)
print(
    "\nInterpretation:"
)
print(
    " - Selective thresholds remain frozen from validation for every seed."
)
print(
    " - Attribution_Fraction is the fraction of flows that actually invoke SHAP."
)
print(
    " - The selective policy is useful only if ANMR/FBR remain stable while "
    "latency decreases and throughput increases."
)
print(
    " - Zero-Day and Adversarial epsilon=0.100 are the critical safety checks."
)


In [ ]:

# BLOCK 18: FINAL SELECTIVE-ATTRIBUTION PIPELINE SANITY CHECK

required_objects = [
    "host_test_df",
    "time_test_df",
    "hard_test_df",
    "df_zero_day_isolated",
    "evaluate_operational_environment",              # 100%-SHAP baseline
    "evaluate_selective_operational_environment",    # final architecture
    "SELECTIVE_BENIGN_DIRECT_PROB_TAU",
    "SELECTIVE_BENIGN_DIRECT_MARGIN_TAU",
    "SELECTIVE_ATTACK_DIRECT_PROB_TAU",
    "SELECTIVE_ATTACK_DIRECT_MARGIN_TAU",
    "constrained_black_box_evasion",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

print("=" * 115)
print("FINAL SELECTIVE-ATTRIBUTION PIPELINE SANITY CHECK")
print("=" * 115)

if missing:
    print(
        "Missing required objects:",
        missing,
    )
else:
    print(
        "All final pipeline dependencies are available."
    )
    print(
        "Final architecture: calibrated probability screen -> selective SHAP/cosine verification -> three-tier routing."
    )
    print(
        "100%-SHAP is retained only as the throughput/safety ablation baseline."
    )
    print("\nFinal experiment set:")
    print("  1. Host-based natural concept drift")
    print("  2. Temporal natural concept drift")
    print("  3. Hard concept drift")
    print("  4. Zero-day attack holdout")
    print("  5. Adversarial evasion robustness")
    print("  6. Selective-attribution multi-seed robustness")

print("=" * 115)
